# Hieroglyphic Translation — Production Pipeline (v4 — P100 + ByT5 Assembly)

**Major changes vs v3:**

1. **GPU: P100 single-card** (was T4×2). Batch sizes retuned for a single 16 GB P100.
   - Removes DDP/multi-GPU overhead → per-step is actually **faster** on one P100 than on two T4s for a ~600 M model.
2. **Columns: new schema** — `raw_transliteration`, `clean_transliteration`, `raw_german`, `clean_german`.
   - Fine-tune runs on `clean_transliteration → clean_german`.
3. **Evaluation on 2,000 samples** — not the full val/test split. All metrics (BLEU, chrF, TER, METEOR, BERTScore, semantic) are computed on a stratified 2k subsample.
4. **ByT5 Sequence Assembly model** *(NEW)* — trained on `character_level.csv` to solve the "`m n r` vs `mnr`" grouping problem. ByT5 is byte-level so it handles the Egyptological Unicode characters (ḥ, ꞽ, ꜣ, ṯ, …) natively.
5. **Repetition dedup** *(NEW)* — post-processing that collapses consecutive character runs (`mnnmn → mnmn`) and consecutive substring repeats (`mnmn → mn`).
6. **Updated pipeline:** Gardner codes → individual transliterations → **ByT5 Assembly** → **repetition dedup** → NLLB → German → (En/Ar pivots).


In [ ]:
# !pip install torch==2.7.1+cu118 torchvision==0.22.1+cu118 --extra-index-url https://download.pytorch.org/whl/cu118 -q

## 1. Environment & GPU detection (single P100)

In [2]:
import torch

print("Torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))

x = torch.randn(512, 512, device='cuda', dtype=torch.float16)
y = x @ x
torch.cuda.synchronize()

print("GPU matmul OK:", torch.isfinite(y).all().item())

Torch: 2.7.1+cu118
GPU: Tesla P100-PCIE-16GB
GPU matmul OK: True


In [3]:
import os, sys, platform

os.environ["CUDA_VISIBLE_DEVICES"]    = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"]  = "false"

ENV = ('kaggle' if os.path.exists('/kaggle')
       else ('colab' if 'COLAB_GPU' in os.environ else 'local'))

print(f'Environment : {ENV}')
print(f'Python      : {sys.version.split()[0]}')
print(f'Platform    : {platform.platform()}')

try:
    import torch
    print(f'Torch       : {torch.__version__}')
    print(f'CUDA avail  : {torch.cuda.is_available()}')

    if torch.cuda.is_available():
        props  = torch.cuda.get_device_properties(0)
        name   = torch.cuda.get_device_name(0)
        mem_gb = props.total_memory / 1e9

        print(f'GPU 0       : {name}  ({mem_gb:.1f} GB)')
        print(f'Compute cap : sm_{props.major}{props.minor}')
        print(f'Num GPUs    : {torch.cuda.device_count()}  (single-GPU mode)')

        HW_FP16 = props.major >= 6
        HW_BF16 = props.major >= 8
        USE_FP16 = HW_FP16 and not HW_BF16
        USE_BF16 = HW_BF16

        print(f'FP16        : {HW_FP16}')
        print(f'BF16        : {HW_BF16}')
        print(f'Precision   : {"bf16" if USE_BF16 else "fp16" if USE_FP16 else "fp32"}')

        torch_ver = tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2])
        if props.major == 6 and torch_ver >= (2, 2):
            print(
                f'WARNING: PyTorch {torch.__version__} does not support sm_{props.major}{props.minor}.\n'
                f'Run: pip install torch==2.1.2+cu118 torchvision==0.16.2+cu118 '
                f'--extra-index-url https://download.pytorch.org/whl/cu118 -q\n'
                f'Then restart the kernel.'
            )
        else:
            _dt = torch.bfloat16 if USE_BF16 else torch.float16 if USE_FP16 else torch.float32
            _x  = torch.ones(4, 4, dtype=_dt, device='cuda')
            assert (_x @ _x).shape == (4, 4)
            del _x; torch.cuda.empty_cache()
            print(f'Smoke test  : PASSED ({_dt})')

except Exception as e:
    print('Torch not available:', e)

Environment : kaggle
Python      : 3.12.12
Platform    : Linux-6.6.113+-x86_64-with-glibc2.35
Torch       : 2.7.1+cu118
CUDA avail  : True
GPU 0       : Tesla P100-PCIE-16GB  (17.1 GB)
Compute cap : sm_60
Num GPUs    : 1  (single-GPU mode)
FP16        : True
BF16        : False
Precision   : fp16
Run: pip install torch==2.1.2+cu118 torchvision==0.16.2+cu118 --extra-index-url https://download.pytorch.org/whl/cu118 -q
Then restart the kernel.


## 2. Install dependencies

In [4]:
import subprocess, sys, importlib

def pip_install(pkgs, quiet=True):
    flag = ['-q'] if quiet else []
    subprocess.check_call([sys.executable, '-m', 'pip', 'install'] + flag + pkgs)

def need(mod, min_ver=None):
    try:
        m = importlib.import_module(mod)
        if min_ver is None: return False
        from packaging.version import Version
        return Version(getattr(m, '__version__', '0')) < Version(min_ver)
    except ImportError:
        return True

wanted = []
if need('sentencepiece', '0.2.0'): wanted.append('sentencepiece>=0.2.0')
if need('transformers', '4.40'):  wanted.append('transformers>=4.40,<4.50')
if need('sentence_transformers', '2.7'): wanted.append('sentence-transformers>=2.7')
if need('sacrebleu', '2.4'):      wanted.append('sacrebleu>=2.4')
if need('sacremoses'):            wanted.append('sacremoses')
if need('bert_score', '0.3.13'):  wanted.append('bert-score>=0.3.13')
if need('evaluate', '0.4'):       wanted.append('evaluate>=0.4')
if need('faiss'):                 wanted.append('faiss-cpu>=1.7.4')
if need('accelerate', '0.29'):    wanted.append('accelerate>=0.29')
if need('datasets', '2.19'):      wanted.append('datasets>=2.19')
if need('nltk', '3.8'):           wanted.append('nltk>=3.8')
if need('rouge_score'):           wanted.append('rouge-score')
if need('Levenshtein'):           wanted.append('python-Levenshtein')

if wanted:
    print(f'Installing: {wanted}')
    pip_install(wanted)
    print('Install done. If you see import errors, restart the kernel ONCE.')
else:
    print('All core deps already present — skipping install.')

import nltk
for pkg in ['punkt', 'punkt_tab', 'wordnet', 'omw-1.4']:
    try: nltk.download(pkg, quiet=True)
    except Exception: pass

Installing: ['sacrebleu>=2.4', 'sacremoses', 'bert-score>=0.3.13', 'evaluate>=0.4', 'faiss-cpu>=1.7.4', 'python-Levenshtein']
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 87.0 MB/s eta 0:00:00
Install done. If you see import errors, restart the kernel ONCE.


## 3. Imports

In [5]:
# Cell 3 — imports (adds ByT5Tokenizer + T5ForConditionalGeneration + Levenshtein)
import os, re, json, math, random, gc, time, warnings, unicodedata, difflib
from pathlib import Path
from dataclasses import dataclass, asdict, field
from typing import List, Dict, Tuple, Optional, Any
from collections import Counter, defaultdict
from contextlib import nullcontext
from itertools import product

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer, AutoModel,
    AutoModelForSeq2SeqLM, AutoConfig,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback, TrainerCallback,
    set_seed,
    pipeline,
    # ByT5 stack for Sequence Assembly
    ByT5Tokenizer, T5ForConditionalGeneration,
    Trainer, TrainingArguments,
)

try:
    import Levenshtein
except ImportError:
    Levenshtein = None

warnings.filterwarnings('ignore')
print('Imports ok.')


Imports ok.


## 4. Central configuration

**What changed:**

| Setting | v3 (old) | v4 (new) | Why |
|---|---|---|---|
| `CUDA_VISIBLE_DEVICES` | `"0,1"` (T4×2) | `"0"` (P100) | Single-GPU → no DDP overhead |
| NLLB `per_device_train_batch_size` | 4 | **8** | P100 has 16 GB and we're not sharing it with DDP |
| NLLB `gradient_accumulation_steps` | 4 | **4** | Effective batch = 8×4 = **32** (same as before) |
| Precision | bf16 (T4 ok) | **fp16** (P100 only supports fp16) | Pascal arch |
| NLLB source/target columns | `translit_norm → german` | `clean_transliteration → clean_german` | New schema |
| Validation size for metrics | full val (~all rows) | **2,000 stratified samples** | Faster, still statistically solid |
| ByT5 sequence assembly | ❌ not present | ✅ trained on `character_level.csv` | Solves the `m n r → mnr` ambiguity |
| Repetition dedup | ❌ not present | ✅ post-processing step | Handles Gardner-induced repeats |


In [6]:
# Cell 4 — configuration (P100 single-GPU + new columns + ByT5)
@dataclass
class Config:
    # -------- dataset paths --------
    data_root        : str = '/kaggle/input/datasets/moazkhaled2003/new-datasettest'
    data_file        : str = 'dataset_cleaned.csv'
    gardiner_file    : str = 'Gardiner_Sign_List.csv'
    intention_file   : str = 'intention_dataset.csv'
    character_file   : str = 'character_level.csv'     # ← used for ByT5 Assembly

    work_dir         : str = '/kaggle/working'
    cache_dir        : str = '/kaggle/working/cache'
    seed             : int = 42

    # -------- column names (NEW SCHEMA) --------
    col_raw_translit   : str = 'raw_transliteration'
    col_clean_translit : str = 'clean_transliteration'
    col_raw_german     : str = 'raw_german'
    col_clean_german   : str = 'clean_german'

    # -------- models --------
    translator_model : str = 'facebook/nllb-200-distilled-600M'
    retriever_model  : str = 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'
    reranker_cometkiwi: str = 'Unbabel/wmt22-cometkiwi-da'
    reranker_fallback : str = 'cross-encoder/stsb-roberta-base'
    sentiment_model  : str = 'cardiffnlp/twitter-roberta-base-sentiment-latest'
    emotion_model    : str = 'j-hartmann/emotion-english-distilroberta-base'
    # ByT5 for Sequence Assembly (character-level, byte-level, handles Egyptological unicode natively)
    assembler_model  : str = 'google/byt5-small'

    # -------- NLLB language tags --------
    src_tag          : str = 'eng_Latn'     # transliteration uses Latin subwords
    de_tag           : str = 'deu_Latn'
    en_tag           : str = 'eng_Latn'
    ar_tag           : str = 'arb_Arab'

    # -------- NLLB training (tuned for single P100 16 GB) --------
    batch_size       : int = 8                  # per-device; was 4 on T4×2
    grad_accum_steps : int = 4
    translator_epochs: int = 5
    early_stopping_patience : int = 3
    min_delta        : float = 1e-4
    learning_rate    : float = 3e-5
    warmup_ratio     : float = 0.06
    weight_decay     : float = 0.01
    max_input_len    : int = 128
    max_target_len   : int = 128
    num_beams        : int = 5
    num_return_sequences : int = 5

    # -------- ByT5 Sequence Assembly training --------
    assembler_epochs           : int = 5
    assembler_batch_size       : int = 16       # ByT5-small is light; fits easily on P100
    assembler_grad_accum       : int = 2
    assembler_max_len          : int = 256
    assembler_lr               : float = 1e-4
    assembler_warmup_steps     : int = 500
    assembler_label_smoothing  : float = 0.1
    assembler_eval_subsample   : int = 2000     # eval on 2k samples of char-level val
    use_assembler              : bool = True

    # -------- Evaluation subsample --------
    eval_subsample_size        : int = 2000     # metrics computed on 2k samples only
    eval_subsample_seed        : int = 42

    # -------- augmentation --------
    use_augmentation : bool  = False
    aug_multiplier   : int   = 1
    p_char_delete    : float = 0.03
    p_char_duplicate : float = 0.02
    p_char_substitute: float = 0.02
    p_seq_crop       : float = 0.10
    min_crop_ratio   : float = 0.6

    # -------- confidence --------
    conf_logprob_weight  : float = 0.35
    conf_retrieval_weight: float = 0.25
    conf_reranker_weight : float = 0.40
    low_confidence_threshold : float = 0.50
    max_fallback_attempts    : int = 4

    # -------- switches --------
    use_retrieval    : bool = True
    use_reranker     : bool = True
    use_self_correction : bool = True
    use_sentiment    : bool = True
    use_emotion      : bool = True
    use_repetition_dedup : bool = True

CFG = Config()
Path(CFG.work_dir).mkdir(parents=True, exist_ok=True)
Path(CFG.cache_dir).mkdir(parents=True, exist_ok=True)

set_seed(CFG.seed); random.seed(CFG.seed); np.random.seed(CFG.seed)
torch.manual_seed(CFG.seed)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(CFG.seed)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
# P100 check — gracefully falls back to fp16 if bf16 not supported
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_FP16 = torch.cuda.is_available() and not USE_BF16

print(f'DEVICE={DEVICE}  bf16={USE_BF16}  fp16={USE_FP16}')
print(f'data_root={CFG.data_root}')
print(f'Effective NLLB batch = {CFG.batch_size}×{CFG.grad_accum_steps} = {CFG.batch_size*CFG.grad_accum_steps}')
print(f'Eval subsample size  = {CFG.eval_subsample_size}')

DEVICE=cuda  bf16=True  fp16=False
data_root=/kaggle/input/datasets/moazkhaled2003/new-datasettest
Effective NLLB batch = 8×4 = 32
Eval subsample size  = 2000


## 5. Utilities

In [7]:
# Cell 5 — utilities (unchanged vs v3)
def free_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.synchronize()
        log(f'GPU free after cleanup: {torch.cuda.mem_get_info()[0]/1e6:.0f} MB')

def tstamp(): return time.strftime('%Y-%m-%d %H:%M:%S')
def log(msg, tag='INFO'): print(f'[{tstamp()}] [{tag}] {msg}')

def save_json(obj, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=str)

def cache_key(task: str, text: str) -> str:
    import hashlib
    return f'{task}::{hashlib.md5(text.encode()).hexdigest()[:12]}'

TRANSLATION_CACHE: Dict[str, Any] = {}
def cache_clear():
    TRANSLATION_CACHE.clear()

def char_edit_distance(a: str, b: str) -> int:
    if Levenshtein is not None:
        return Levenshtein.distance(a, b)
    # fallback DP
    if len(a) < len(b): a, b = b, a
    if not b: return len(a)
    prev = list(range(len(b)+1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(cur[-1]+1, prev[j]+1, prev[j-1]+(ca!=cb)))
        prev = cur
    return prev[-1]

def unified_diff(a: str, b: str) -> str:
    '''Inline diff — `[-…-]` deletions, `{+…+}` insertions.'''
    sm = difflib.SequenceMatcher(None, a, b)
    out = []
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == 'equal':   out.append(a[i1:i2])
        elif tag == 'delete': out.append(f'[-{a[i1:i2]}-]')
        elif tag == 'insert': out.append(f'{{+{b[j1:j2]}+}}')
        elif tag == 'replace':
            out.append(f'[-{a[i1:i2]}-]{{+{b[j1:j2]}+}}')
    return ''.join(out)

log('utilities ready')


[2026-04-24 12:02:03] [INFO] utilities ready


## 6. Data loading — **new column schema**

Dataset now has four columns:
- `raw_transliteration`
- `clean_transliteration`  ← used as source for fine-tuning
- `raw_german`
- `clean_german`           ← used as target for fine-tuning

The loader handles both the new schema and the old one (backward-compatible rename).


In [8]:
# Cell 6 — load dataset with new schema
def load_main_corpus(cfg: Config) -> pd.DataFrame:
    p = Path(cfg.data_root) / cfg.data_file
    if not p.exists():
        log(f'{p} missing — using 8-row seed (NOT trainable, pipeline test only).', 'WARN')
        seed = [
            {'raw_transliteration': 'ḥtp-di-nsw',  'clean_transliteration': 'ḥtp di nsw',
             'raw_german': 'ein Opfer, das der König gibt', 'clean_german': 'ein Opfer das der König gibt'},
            {'raw_transliteration': 'nsw-bꞽtꞽ',   'clean_transliteration': 'nsw bꞽtꞽ',
             'raw_german': 'König von Ober- und Unterägypten', 'clean_german': 'König von Ober und Unterägypten'},
            {'raw_transliteration': 'sꜣ-rꜥ',       'clean_transliteration': 'sꜣ rꜥ',
             'raw_german': 'Sohn des Re', 'clean_german': 'Sohn des Re'},
            {'raw_transliteration': 'nṯr nfr',     'clean_transliteration': 'nṯr nfr',
             'raw_german': 'der vollkommene Gott', 'clean_german': 'der vollkommene Gott'},
            {'raw_transliteration': 'ꜥꜣ ḫrw',      'clean_transliteration': 'ꜥꜣ ḫrw',
             'raw_german': 'groß an Stimme', 'clean_german': 'groß an Stimme'},
            {'raw_transliteration': 'ḏrt',         'clean_transliteration': 'ḏrt',
             'raw_german': 'die Hand', 'clean_german': 'die Hand'},
            {'raw_transliteration': 'nṯrw',        'clean_transliteration': 'nṯrw',
             'raw_german': 'die Götter', 'clean_german': 'die Götter'},
            {'raw_transliteration': 'ꞽr nfr',      'clean_transliteration': 'ꞽr nfr',
             'raw_german': 'tue, was gut ist', 'clean_german': 'tue was gut ist'},
        ]
        return pd.DataFrame(seed)

    df = pd.read_csv(p)
    # Backward-compat: if the CSV still uses the old v3 schema, rename to new schema
    legacy_map = {
        'translit_raw'        : cfg.col_raw_translit,
        'raw_text'            : cfg.col_raw_translit,
        'translit_clean'      : cfg.col_clean_translit,
        'clean_text'          : cfg.col_clean_translit,
        'german'              : cfg.col_clean_german,
        'german_translation'  : cfg.col_clean_german,
        'raw_german_text'     : cfg.col_raw_german,
    }
    for src, dst in legacy_map.items():
        if src in df.columns and dst not in df.columns:
            df = df.rename(columns={src: dst})
            log(f'Renamed legacy column {src} → {dst}')

    # Ensure all four expected columns exist (fill missing with empty strings)
    for c in [cfg.col_raw_translit, cfg.col_clean_translit,
              cfg.col_raw_german,  cfg.col_clean_german]:
        if c not in df.columns:
            df[c] = ''
            log(f'Column {c} not found — created empty.', 'WARN')

    before = len(df)
    df = df[(df[cfg.col_clean_translit].astype(str).str.strip() != '') &
            (df[cfg.col_clean_german].astype(str).str.strip()    != '')].fillna('').copy()
    log(f'Loaded {len(df)}/{before} usable parallel rows from {p}')
    log(f'Columns: {list(df.columns)}')
    return df.reset_index(drop=True)

def load_gardiner_table(cfg: Config) -> pd.DataFrame:
    p = Path(cfg.data_root) / cfg.gardiner_file
    if not p.exists():
        log(f'{p} missing — Gardiner lookup disabled.', 'WARN')
        return pd.DataFrame(columns=['code', 'transliterations', 'type'])
    return pd.read_csv(p).fillna('')

def load_intention_table(cfg: Config) -> pd.DataFrame:
    p = Path(cfg.data_root) / cfg.intention_file
    if not p.exists():
        log(f'{p} missing — intention classifier disabled.', 'WARN')
        return pd.DataFrame(columns=['intention_en', 'intention_ar', 'keywords'])
    return pd.read_csv(p).fillna('')

def load_character_level(cfg: Config) -> pd.DataFrame:
    '''NEW — character_level.csv: spaced characters → assembled form (ByT5 training).'''
    p = Path(cfg.data_root) / cfg.character_file
    if not p.exists():
        log(f'{p} missing — ByT5 Sequence Assembly training will be skipped.', 'WARN')
        return pd.DataFrame(columns=['characters', 'clean_text'])
    df = pd.read_csv(p).fillna('')
    # Accept either ('characters', 'clean_text') or ('input_text', 'target_text')
    if 'input_text' in df.columns and 'target_text' in df.columns and 'characters' not in df.columns:
        df = df.rename(columns={'input_text': 'characters', 'target_text': 'clean_text'})
    for c in ['characters', 'clean_text']:
        if c not in df.columns: df[c] = ''
    df = df[(df['characters'].astype(str).str.strip() != '') &
            (df['clean_text'].astype(str).str.strip()  != '')].reset_index(drop=True)
    log(f'Loaded character_level.csv: {len(df)} char-assembly pairs')
    return df

DF          = load_main_corpus(CFG)
GARDINER    = load_gardiner_table(CFG)
INTENTION   = load_intention_table(CFG)
CHAR_LEVEL  = load_character_level(CFG)

IS_SEED = len(DF) <= 20
print(f'Main corpus    : {len(DF):>6d} rows')
print(f'Gardiner table : {len(GARDINER):>6d} signs')
print(f'Intentions     : {len(INTENTION):>6d} categories')
print(f'Character-level: {len(CHAR_LEVEL):>6d} pairs (for ByT5 Assembly)')
if IS_SEED:
    log('SEED CORPUS — metrics will be flagged INSUFFICIENT_DATA.', 'WARN')
DF.head(3)


[2026-04-24 12:02:04] [INFO] Loaded 113463/113463 usable parallel rows from /kaggle/input/datasets/moazkhaled2003/new-datasettest/dataset_cleaned.csv
[2026-04-24 12:02:04] [INFO] Columns: ['raw_transliteration', 'clean_transliteration', 'raw_german', 'clean_german']
[2026-04-24 12:02:05] [INFO] Loaded character_level.csv: 113456 char-assembly pairs
Main corpus    : 113463 rows
Gardiner table :   7028 signs
Intentions     :    139 categories
Character-level: 113456 pairs (for ByT5 Assembly)


,raw_transliteration,clean_transliteration,raw_german,clean_german
0,nḏ (w)di̯ r =s,nḏ wdi̯ r s,"(es) werde zerrieben, (es) werde darauf gelegt.","werde zerrieben, werde darauf gelegt."
1,n ṯw ꞽm =sn,n ṯw ꞽm sn,Du gehörst nicht zu ihnen.,Du gehörst nicht zu ihnen.
2,ḫꜣ m tʾ ḥnq.t kꜣ(.PL) ꜣpd(.PL) n ꞽmꜣḫ ꞽm.ꞽ-rʾ-...,ḫꜣ m tʾ ḥnqt kꜣ ꜣpd n ꞽmꜣḫ ꞽmꞽ rʾ šnꜥ ꞽmn m ḥꜣ...,"Tausend an Brot, Bier, Rindern und Geflügel fü...","Tausend an Brot, Bier, Rindern und Geflügel fü..."


## 7. Preprocessing — ASCII → Egyptological (idempotent)

Same logic as v3 (the v2 `.lower()` collision bug is fixed), but now operates on the
new column names (`raw_transliteration`, `clean_transliteration`, etc.).


In [9]:
# Cell 7 — normalization (ASCII MdC → Egyptological Unicode, idempotent)
def clean_text(s: str) -> str:
    s = unicodedata.normalize('NFC', str(s))
    s = re.sub(r'\s+', ' ', s).strip()
    return s

MDC_TO_EGYPTO = {
    'A': 'ꜣ', 'i': 'ꞽ', 'y': 'y', 'a': 'ꜥ',
    'H': 'ḥ', 'x': 'ḫ', 'X': 'ẖ', 'S': 'š',
    'T': 'ṯ', 'D': 'ḏ',
}
APOSTROPHES = {'ʾ': "'", '\u2018': "'", '\u2019': "'"}
EGYPTO_SIGNATURES = set('ꜣꞽꜥḥḫẖšṯḏ')

def looks_like_mdc(s: str) -> bool:
    if any(c in EGYPTO_SIGNATURES for c in s): return False
    return bool(re.search(r'[AHSTDXxia]', s))

def mdc_to_egypto(s: str) -> str:
    return ''.join(MDC_TO_EGYPTO.get(c, c) for c in s)

def normalize_translit(s: str) -> str:
    '''Always returns Egyptological form — matches training data.'''
    s = clean_text(s)
    for k, v in APOSTROPHES.items(): s = s.replace(k, v)
    if looks_like_mdc(s):
        s = mdc_to_egypto(s)
    s = re.sub(r'[.()\[\]<>\-]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

# smoke tests
assert normalize_translit('HAty ib') == normalize_translit('ḥꜣty ꞽb')
assert normalize_translit('sA ra') == 'sꜣ rꜥ'
assert normalize_translit('ḥtp di nsw') == 'ḥtp di nsw'
assert normalize_translit('xrp-ab') == 'ḫrp ꜥb'
assert normalize_translit('nTr nfr') == 'nṯr nfr'
print('Normalizer smoke-tests passed.')

# Apply on new column schema
DF[CFG.col_raw_translit]   = DF[CFG.col_raw_translit].astype(str).apply(clean_text)
DF[CFG.col_clean_translit] = DF[CFG.col_clean_translit].astype(str).apply(clean_text)
DF[CFG.col_raw_german]     = DF[CFG.col_raw_german].astype(str).apply(clean_text)
DF[CFG.col_clean_german]   = DF[CFG.col_clean_german].astype(str).apply(clean_text)

# Normalized transliteration used downstream (Gardiner lookup, FAISS, etc.)
DF['translit_norm'] = DF[CFG.col_clean_translit].apply(normalize_translit)

# If raw_german is empty, fall back to clean_german for downstream use
mask_empty_raw_de = DF[CFG.col_raw_german].astype(str).str.strip() == ''
DF.loc[mask_empty_raw_de, CFG.col_raw_german] = DF.loc[mask_empty_raw_de, CFG.col_clean_german]

DF = DF[(DF['translit_norm'].str.len() > 0) &
        (DF[CFG.col_clean_german].str.len() > 0)].reset_index(drop=True)

print(f'After cleaning: {len(DF)} rows')
print()
print('Sample normalized rows:')
cols_show = [CFG.col_raw_translit, 'translit_norm', CFG.col_clean_translit,
             CFG.col_clean_german]
print(DF[cols_show].head(5).to_string(index=False))

Normalizer smoke-tests passed.
After cleaning: 111637 rows

Sample normalized rows:
                                                    raw_transliteration                                           translit_norm                                   clean_transliteration                                                                                                            clean_german
                                                         nḏ (w)di̯ r =s                                             nḏ wdi̯ r s                                             nḏ wdi̯ r s                                                                                   werde zerrieben, werde darauf gelegt.
                                                            n ṯw ꞽm =sn                                              n ṯw ꞽm sn                                              n ṯw ꞽm sn                                                                                              Du gehörst nicht zu ihnen.
ḫꜣ m

## 8. Train / Val / Test split — leakage-guarded on normalized form

In [10]:
# Cell 8 — stratified split + dedup (unchanged logic, new column name for german)
from sklearn.model_selection import train_test_split

def length_bucket(s: str) -> str:
    n = len(s.split())
    if n <= 3:  return 'short'
    if n <= 8:  return 'medium'
    if n <= 20: return 'long'
    return 'xlong'

DF['len_bucket'] = DF['translit_norm'].apply(length_bucket)

def split_with_dedup(df, test_size=0.10, val_size=0.10, seed=42):
    trainval, test = train_test_split(df, test_size=test_size, random_state=seed,
                                      stratify=df['len_bucket'])
    train, val = train_test_split(trainval, test_size=val_size/(1-test_size),
                                  random_state=seed, stratify=trainval['len_bucket'])
    train_set = set(train['translit_norm'])
    b_test, b_val = len(test), len(val)
    test = test[~test['translit_norm'].isin(train_set)].reset_index(drop=True)
    val  = val[~val['translit_norm'].isin(train_set)].reset_index(drop=True)
    removed = (b_test - len(test)) + (b_val - len(val))
    if removed: log(f'Leakage guard removed {removed} overlap rows from val/test.', 'WARN')
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)

if len(DF) < 30:
    TRAIN, VAL, TEST = DF.copy(), DF.copy(), DF.copy()
    log('Tiny corpus — train=val=test (smoke mode).', 'WARN')
else:
    TRAIN, VAL, TEST = split_with_dedup(DF)

print(f'train={len(TRAIN)}  val={len(VAL)}  test={len(TEST)}')
print(TRAIN['len_bucket'].value_counts().to_string())

[2026-04-24 12:02:09] [WARN] Leakage guard removed 6079 overlap rows from val/test.
train=89309  val=8121  test=8128
len_bucket
medium    37343
long      26356
short     18581
xlong      7029


## 8.5. **NEW** — stratified 2,000-sample evaluation subsample

Metrics (BLEU/chrF/TER/METEOR/BERTScore/semantic) are **expensive** on 5k+ rows, especially
BERTScore which runs a 1 B-param model per pair. We take a **stratified** 2,000-sample
subset of `VAL` / `TEST` so each length bucket is represented proportionally.


In [11]:
# Cell 8.5 — stratified eval subsample (NEW)
def stratified_subsample(df: pd.DataFrame, n: int, seed: int = 42,
                         strat_col: str = 'len_bucket') -> pd.DataFrame:
    '''Return at most n rows, stratified by strat_col.
       If df is smaller than n, return the whole df.'''
    if len(df) <= n:
        log(f'Subsample skipped: have {len(df)} ≤ target {n}')
        return df.reset_index(drop=True)
    if strat_col not in df.columns:
        return df.sample(n=n, random_state=seed).reset_index(drop=True)

    # Proportional allocation per bucket
    rng = np.random.RandomState(seed)
    buckets = df[strat_col].value_counts(normalize=True)
    parts = []
    remaining = n
    for b, frac in buckets.items():
        sub = df[df[strat_col] == b]
        k   = min(len(sub), int(round(frac * n)))
        if k > 0:
            parts.append(sub.sample(n=k, random_state=seed))
            remaining -= k
    out = pd.concat(parts, ignore_index=True)
    # Fill any shortfall from rounding
    if remaining > 0 and len(out) < len(df):
        rest = df.drop(index=out.index, errors='ignore')
        if len(rest) > 0:
            extra = rest.sample(n=min(remaining, len(rest)), random_state=seed+1)
            out = pd.concat([out, extra], ignore_index=True)
    out = out.sample(frac=1.0, random_state=seed).reset_index(drop=True)  # shuffle
    return out.head(n)

# Build the evaluation subsamples up-front
VAL_EVAL  = stratified_subsample(VAL,  CFG.eval_subsample_size, CFG.eval_subsample_seed)
TEST_EVAL = stratified_subsample(TEST, CFG.eval_subsample_size, CFG.eval_subsample_seed)

log(f'VAL_EVAL  subsample: {len(VAL_EVAL)} rows (of {len(VAL)})')
log(f'TEST_EVAL subsample: {len(TEST_EVAL)} rows (of {len(TEST)})')
print('VAL_EVAL length-bucket distribution:')
print(VAL_EVAL['len_bucket'].value_counts().to_string())


[2026-04-24 12:02:14] [INFO] VAL_EVAL  subsample: 2000 rows (of 8121)
[2026-04-24 12:02:14] [INFO] TEST_EVAL subsample: 2000 rows (of 8128)
VAL_EVAL length-bucket distribution:
len_bucket
medium    848
long      709
short     237
xlong     206


## 9. Gardiner lookup — context-aware variant selection

In [12]:
# Cell 9 — Gardiner → transliteration (context-aware, unchanged vs v3)

GARDINER = pd.read_csv(
    Path(CFG.data_root) / CFG.gardiner_file
).fillna('') if (Path(CFG.data_root) / CFG.gardiner_file).exists() else GARDINER
print(f'Gardiner CSV loaded: {len(GARDINER)} rows')

TRAIN_TOK_COUNTER: Counter = Counter()
for s in DF['translit_norm']:
    TRAIN_TOK_COUNTER.update(s.split())

def build_ngram_context(df: pd.DataFrame) -> Dict[str, Counter]:
    ctx: Dict[str, Counter] = defaultdict(Counter)
    for s in df['translit_norm']:
        toks = s.split()
        for i in range(1, len(toks)):
            ctx[toks[i]][toks[i-1]] += 1
    return ctx

NGRAM_CTX = build_ngram_context(DF)
log(f'Bigram context built: {len(NGRAM_CTX)} unique tokens')

def build_gardiner_map(df: pd.DataFrame) -> Dict[str, List[str]]:
    mp: Dict[str, List[str]] = {}
    for _, r in df.iterrows():
        code = str(r.get('code', '')).strip()
        trl  = str(r.get('transliterations', '')).strip()
        if not code or not trl: continue
        variants = [normalize_translit(v.strip()) for v in trl.split('|') if v.strip()]
        variants = list(dict.fromkeys(variants))
        variants.sort(key=lambda v: -TRAIN_TOK_COUNTER.get(v, 0))
        mp[code] = variants
    return mp

GARDINER_MAP: Dict[str, List[str]] = build_gardiner_map(GARDINER)
log(f'Gardiner lookup: {len(GARDINER_MAP)} codes (multi-variant aware)')

GARDINER_RE = re.compile(r'[A-Z]{1,2}[a-z]?\d{1,3}[A-Za-z]?')

def tokenize_gardiner(s: str) -> List[str]:
    s = clean_text(s).replace(',', ' ').replace('-', ' ').replace('|', ' ')
    return [t for t in s.split() if GARDINER_RE.fullmatch(t)]

def gardiner_to_translit(codes: List[str], mode: str = 'top') -> str:
    out = []
    for c in codes:
        variants = GARDINER_MAP.get(c, [])
        if not variants:
            out.append(c.lower()); continue
        if mode == 'top':
            out.append(variants[0])
        else:
            out.append('|'.join(variants))
    return ' '.join(out)

def gardiner_to_translit_ctx(codes: List[str]) -> str:
    out = []
    for i, c in enumerate(codes):
        variants = GARDINER_MAP.get(c, [])
        if not variants:
            out.append(c.lower()); continue
        if i == 0 or not out:
            out.append(variants[0]); continue
        prev_word    = out[-1]
        best_variant = variants[0]; best_score = -1
        for v in variants:
            ctx_score    = NGRAM_CTX[v][prev_word]
            global_score = TRAIN_TOK_COUNTER.get(v, 0)
            score        = (ctx_score * 3) + global_score
            if score > best_score:
                best_score, best_variant = score, v
        out.append(best_variant)
    return ' '.join(out)

def gardiner_expand(codes: List[str], max_variants: int = 8) -> List[str]:
    per_code = [GARDINER_MAP.get(c, [c.lower()])[:3] or [c.lower()] for c in codes]
    cands    = [' '.join(combo) for combo in product(*per_code)]
    return cands[:max_variants]

if GARDINER_MAP:
    demo  = 'G17 M18 F34'
    codes = tokenize_gardiner(demo)
    print(f'{demo}  → (top)    {gardiner_to_translit(codes)}')
    print(f'{demo}  → (ctx)    {gardiner_to_translit_ctx(codes)}')
    print(f'{demo}  → (expand) {gardiner_expand(codes)[:3]}')


Gardiner CSV loaded: 7028 rows
[2026-04-24 12:02:18] [INFO] Bigram context built: 23423 unique tokens
[2026-04-24 12:02:18] [INFO] Gardiner lookup: 1658 codes (multi-variant aware)
G17 M18 F34  → (top)    m ꞽꞽ ꞽb
G17 M18 F34  → (ctx)    m ꞽꞽ ꞽb
G17 M18 F34  → (expand) ['m ꞽꞽ ꞽb', 'm ꞽꞽ ḥꜣty']


## 10. **NEW** — ByT5 Sequence Assembly training on `character_level.csv`

**The problem this solves:**
If the NLLB was trained on `ḥtp di nsw mnr` and you give Gardiner codes that produce
`htp di nsw m n r` or `htp-di nswa m n r`, NLLB has **no way** to know whether
`m n r` should become `mnr`, `mn r`, or `m nr`. Spacing in Egyptological transliteration
is meaningful — it separates words. So we need a model that takes spaced characters
and produces correctly-grouped tokens.

**The solution:** a small ByT5 model (byte-level) trained on `character_level.csv`:

| characters            | clean_text    |
|-----------------------|---------------|
| `n ḏ w d i ̯ r s`     | `nḏ wdi̯ r s` |
| `n ṯ w ꞽ m s n`       | `n ṯw ꞽm sn`   |
| `ꜥ ḥ ꜥ`               | `ꜥḥꜥ`          |

ByT5 is **byte-level** so it handles all the Egyptological Unicode characters (ḥ, ꞽ, ꜣ, ṯ, ḏ, š, ẖ, ḫ, ꜥ) natively with **no tokenizer surprises**. This is exactly what the reference notebook does — we import the same architecture and training config.

**Position in the pipeline:**
```
Gardiner codes  →  individual transliterations (spaced)  →  ByT5 Assembly (grouped)  →  repetition dedup  →  NLLB → German
```


In [13]:
# Cell 10 — ByT5 Sequence Assembly training (FAST EVAL VERSION)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ASSEMBLER_CKPT = f'{CFG.work_dir}/byt5_assembler'
EXTERNAL_CKPT  = '/kaggle/input/notebooks/menozkhalifaz/notebook4f43a7d4f5/byt5_assembler'

def find_latest_checkpoint(ckpt_dir):
    from pathlib import Path

    # ① شوف الأول في الـ notebook القديم
    ext = Path(EXTERNAL_CKPT)
    if ext.exists():
        ckpts = sorted(
            [d for d in ext.iterdir() if d.is_dir() and d.name.startswith('checkpoint-')],
            key=lambda x: int(x.name.split('-')[-1])
        )
        if ckpts:
            log(f'Found external checkpoint: {ckpts[-1]}')
            return str(ckpts[-1])

    # ② لو مش موجود، شوف في working dir
    p = Path(ckpt_dir)
    if not p.exists():
        return None
    ckpts = sorted(
        [d for d in p.iterdir() if d.is_dir() and d.name.startswith('checkpoint-')],
        key=lambda x: int(x.name.split('-')[-1])
    )
    return str(ckpts[-1]) if ckpts else None

def train_byt5_assembler():
    if len(CHAR_LEVEL) < 50:
        log('character_level.csv too small — skipping ByT5 assembly training.', 'WARN')
        return

    log(f'Training ByT5 Assembler  out={ASSEMBLER_CKPT}')
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    resume_ckpt = find_latest_checkpoint(ASSEMBLER_CKPT)
    if resume_ckpt:
        log(f'Resuming from checkpoint: {resume_ckpt}')
        tok = ByT5Tokenizer.from_pretrained(resume_ckpt, cache_dir=CFG.cache_dir)
        mdl = T5ForConditionalGeneration.from_pretrained(resume_ckpt,
                                                         cache_dir=CFG.cache_dir)
    else:
        log('No checkpoint found — starting from scratch.')
        tok = ByT5Tokenizer.from_pretrained(CFG.assembler_model, cache_dir=CFG.cache_dir)
        mdl = T5ForConditionalGeneration.from_pretrained(CFG.assembler_model,
                                                         cache_dir=CFG.cache_dir)

    # Split character_level into train / val
    rng = np.random.RandomState(CFG.seed)
    idx = np.arange(len(CHAR_LEVEL))
    rng.shuffle(idx)
    n_val   = max(200, int(0.1 * len(CHAR_LEVEL)))
    val_idx = idx[:n_val]
    tr_idx  = idx[n_val:]
    tr_df = CHAR_LEVEL.iloc[tr_idx].reset_index(drop=True)
    va_df = CHAR_LEVEL.iloc[val_idx].reset_index(drop=True)

    EVAL_SUBSAMPLE = getattr(CFG, 'assembler_eval_subsample', 500)
    EVAL_SUBSAMPLE = min(EVAL_SUBSAMPLE, 500)
    if len(va_df) > EVAL_SUBSAMPLE:
        va_df = va_df.sample(n=EVAL_SUBSAMPLE,
                             random_state=CFG.seed).reset_index(drop=True)

    log(f'ByT5 train: {len(tr_df)}  val_eval: {len(va_df)}')

    class ByteAssemblyDataset(Dataset):
        def __init__(self, df, tok, max_len):
            self.df   = df.reset_index(drop=True)
            self.tok  = tok
            self.maxl = max_len
        def __len__(self): return len(self.df)
        def __getitem__(self, i):
            r = self.df.iloc[i]
            enc = self.tok(text=str(r['characters']),
                           text_target=str(r['clean_text']),
                           max_length=self.maxl, truncation=True, padding=False)
            return enc

    tr_ds = ByteAssemblyDataset(tr_df, tok, CFG.assembler_max_len)
    va_ds = ByteAssemblyDataset(va_df, tok, CFG.assembler_max_len)
    collator = DataCollatorForSeq2Seq(tokenizer=tok, model=mdl, padding=True)

    args = TrainingArguments(
        output_dir                  = ASSEMBLER_CKPT,
        per_device_train_batch_size = CFG.assembler_batch_size,
        per_device_eval_batch_size  = max(4, CFG.assembler_batch_size // 2),
        gradient_accumulation_steps = CFG.assembler_grad_accum,
        num_train_epochs            = CFG.assembler_epochs,
        optim                       = 'adafactor',
        learning_rate               = CFG.assembler_lr,
        lr_scheduler_type           = 'cosine',
        label_smoothing_factor      = CFG.assembler_label_smoothing,
        warmup_steps                = CFG.assembler_warmup_steps,
        weight_decay                = 0.01,
        logging_steps               = 50,
        eval_strategy               = 'steps',
        eval_steps                  = 500,
        save_strategy               = 'steps',
        save_steps                  = 500,
        metric_for_best_model       = 'eval_loss',
        greater_is_better           = False,
        save_total_limit            = 2,
        load_best_model_at_end      = True,
        report_to                   = 'none',
        fp16                        = USE_FP16,
        bf16                        = USE_BF16,
        seed                        = CFG.seed,
        dataloader_num_workers      = 2,
        gradient_checkpointing      = True,
        gradient_checkpointing_kwargs = {'use_reentrant': False},
    )

    try:
        trainer = Trainer(
            model            = mdl,
            args             = args,
            train_dataset    = tr_ds,
            eval_dataset     = va_ds,
            data_collator    = collator,
            processing_class = tok,
        )
    except TypeError:
        trainer = Trainer(
            model         = mdl,
            args          = args,
            train_dataset = tr_ds,
            eval_dataset  = va_ds,
            data_collator = collator,
            tokenizer     = tok,
        )

    trainer.train(resume_from_checkpoint=resume_ckpt)

    trainer.save_model(ASSEMBLER_CKPT)
    tok.save_pretrained(ASSEMBLER_CKPT)
    save_json(trainer.state.log_history, f'{ASSEMBLER_CKPT}/train_history.json')
    log(f'ByT5 Assembler saved to {ASSEMBLER_CKPT}')

    log('Computing final metrics (EM / Levenshtein / Combined) on validation set...')
    mdl.eval()
    device = next(mdl.parameters()).device

    ALPHA = 0.85
    all_preds, all_labels = [], []

    BATCH = max(4, CFG.assembler_batch_size // 2)
    with torch.no_grad():
        for start in range(0, len(va_df), BATCH):
            batch = va_df.iloc[start:start + BATCH]
            inputs = tok(
                [str(x) for x in batch['characters'].tolist()],
                max_length=CFG.assembler_max_len,
                truncation=True, padding=True, return_tensors='pt',
            ).to(device)

            out = mdl.generate(
                **inputs,
                max_length=CFG.assembler_max_len,
                num_beams=1,
            )
            decoded = tok.batch_decode(out, skip_special_tokens=True)
            all_preds.extend([p.strip() for p in decoded])
            all_labels.extend([str(x).strip() for x in batch['clean_text'].tolist()])

    em = [int(p == l) for p, l in zip(all_preds, all_labels)]
    if Levenshtein is not None:
        lev = [Levenshtein.ratio(p, l) for p, l in zip(all_preds, all_labels)]
    else:
        lev = [1.0 if p == l else 0.0 for p, l in zip(all_preds, all_labels)]

    avg_em, avg_lev = float(np.mean(em)), float(np.mean(lev))
    final_metrics = {
        'exact_match'           : avg_em,
        'levenshtein_similarity': avg_lev,
        'combined_score'        : ALPHA * avg_em + (1 - ALPHA) * avg_lev,
        'n_samples'             : len(all_preds),
    }
    log(f'Final metrics → EM={avg_em:.4f}  Lev={avg_lev:.4f}  '
        f'Combined={final_metrics["combined_score"]:.4f}')
    save_json(final_metrics, f'{ASSEMBLER_CKPT}/final_metrics.json')


if CFG.use_assembler and len(CHAR_LEVEL) >= 50:
    if not Path(ASSEMBLER_CKPT, 'config.json').exists():
        train_byt5_assembler()
        free_cuda()
    else:
        log(f'Found existing ByT5 checkpoint at {ASSEMBLER_CKPT} — skipping.')
else:
    log('ByT5 Assembler disabled or insufficient data.', 'WARN')

[2026-04-24 12:02:42] [INFO] Training ByT5 Assembler  out=/kaggle/working/byt5_assembler
[2026-04-24 12:02:42] [INFO] Found external checkpoint: /kaggle/input/notebooks/menozkhalifaz/notebook4f43a7d4f5/byt5_assembler/checkpoint-15955
[2026-04-24 12:02:42] [INFO] Resuming from checkpoint: /kaggle/input/notebooks/menozkhalifaz/notebook4f43a7d4f5/byt5_assembler/checkpoint-15955


Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


[2026-04-24 12:02:43] [INFO] ByT5 train: 102111  val_eval: 500


Could not locate the best model at /kaggle/working/byt5_assembler/checkpoint-14000/pytorch_model.bin, if you are running a distributed training on multiple nodes, you should activate `--save_on_each_node`.


Step,Training Loss,Validation Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[2026-04-24 12:02:54] [INFO] ByT5 Assembler saved to /kaggle/working/byt5_assembler
[2026-04-24 12:02:54] [INFO] Computing final metrics (EM / Levenshtein / Combined) on validation set...
[2026-04-24 12:11:22] [INFO] Final metrics → EM=0.6720  Lev=0.9870  Combined=0.7193
[2026-04-24 12:11:23] [INFO] GPU free after cleanup: 16722 MB


## 11. **NEW** — ByT5 Assembly inference (`assemble_spaced_chars`)

Takes a spaced-character string like `"m n r"` and returns the grouped form like `"mnr"`.
If the ByT5 checkpoint isn't available, we fall back to a heuristic (simple whitespace collapse).


In [14]:
# Cell 11 — ByT5 Assembly inference + 30-sample side-by-side comparison
ASSEMBLER_TOK = None
ASSEMBLER_MDL = None

def load_assembler():
    global ASSEMBLER_TOK, ASSEMBLER_MDL
    if ASSEMBLER_MDL is not None: return
    if not Path(ASSEMBLER_CKPT, 'config.json').exists():
        log('No ByT5 Assembly checkpoint — heuristic fallback will be used.', 'WARN')
        return
    log(f'Loading ByT5 Assembler from {ASSEMBLER_CKPT}')
    ASSEMBLER_TOK = ByT5Tokenizer.from_pretrained(ASSEMBLER_CKPT)
    ASSEMBLER_MDL = T5ForConditionalGeneration.from_pretrained(ASSEMBLER_CKPT).to(DEVICE).eval()

@torch.no_grad()
def assemble_spaced_chars(text: str, max_len: int = 256) -> str:
    '''Single-input version (used by the pipeline).'''
    if not text.strip():
        return text
    load_assembler()
    if ASSEMBLER_MDL is None:
        return re.sub(r'\s+', '', text) if len(text.replace(' ', '')) <= 8 else text
    enc = ASSEMBLER_TOK(text, return_tensors='pt', truncation=True,
                        max_length=max_len).to(DEVICE)
    out = ASSEMBLER_MDL.generate(**enc, max_length=max_len, num_beams=4,
                                 early_stopping=True)
    decoded = ASSEMBLER_TOK.decode(out[0], skip_special_tokens=True)
    return decoded.strip()

@torch.no_grad()
def assemble_spaced_chars_batch(texts, max_len=256, batch_size=16):
    '''Batched version — أسرع بكتير لو عايز تمرر 30+ مثال.'''
    load_assembler()
    if ASSEMBLER_MDL is None:
        return [re.sub(r'\s+', '', t) if len(t.replace(' ', '')) <= 8 else t
                for t in texts]
    results = []
    for start in range(0, len(texts), batch_size):
        batch = [t if t.strip() else ' ' for t in texts[start:start+batch_size]]
        enc = ASSEMBLER_TOK(batch, return_tensors='pt', truncation=True,
                            max_length=max_len, padding=True).to(DEVICE)
        out = ASSEMBLER_MDL.generate(**enc, max_length=max_len, num_beams=4,
                                     early_stopping=True)
        decoded = ASSEMBLER_TOK.batch_decode(out, skip_special_tokens=True)
        results.extend([d.strip() for d in decoded])
    return results

# ═══════════════════════════════════════════════════════════════════════
#  30-sample side-by-side comparison on the ByT5 validation set
# ═══════════════════════════════════════════════════════════════════════
if Path(ASSEMBLER_CKPT, 'config.json').exists() and len(CHAR_LEVEL) > 0:
    # Reconstruct the EXACT validation split Cell 10 used (same seed = same split)
    rng = np.random.RandomState(CFG.seed)
    idx = np.arange(len(CHAR_LEVEL))
    rng.shuffle(idx)
    n_val   = max(200, int(0.1 * len(CHAR_LEVEL)))
    val_idx = idx[:n_val]
    val_df  = CHAR_LEVEL.iloc[val_idx].reset_index(drop=True)

    N_SHOW = 30
    sample = val_df.head(N_SHOW) if len(val_df) >= N_SHOW else val_df

    log(f'Running ByT5 inference on {len(sample)} validation samples...')
    inputs = [str(x) for x in sample['characters'].tolist()]
    golds  = [str(x) for x in sample['clean_text'].tolist()]
    t0 = time.time()
    preds = assemble_spaced_chars_batch(inputs, batch_size=16)
    log(f'Done in {time.time()-t0:.1f}s')

    # Per-example metrics
    def lev_ratio(a, b):
        if Levenshtein is not None:
            return Levenshtein.ratio(a, b)
        if not a and not b: return 1.0
        return 1.0 - char_edit_distance(a, b) / max(len(a), len(b), 1)

    def truncate(s, n=40):
        return s if len(s) <= n else s[:n-1] + '…'

    rows = []
    for i, (inp, gold, pred) in enumerate(zip(inputs, golds, preds), 1):
        em  = int(pred == gold)
        lev = lev_ratio(pred, gold)
        rows.append({
            '#'    : i,
            'INPUT (spaced chars)': truncate(inp, 35),
            'GOLD'                : truncate(gold, 30),
            'PRED (ByT5)'         : truncate(pred, 30),
            'EM'                  : '✓' if em else '✗',
            'Lev'                 : f'{lev:.2f}',
        })
    comp_df = pd.DataFrame(rows)

    print('\n' + '═' * 110)
    print(f' ByT5 Assembler — Predicted vs Actual ({len(sample)} samples from validation set)')
    print('═' * 110)
    with pd.option_context('display.max_colwidth', 40,
                           'display.width', 200,
                           'display.max_rows', None):
        print(comp_df.to_string(index=False))

    # Summary
    ems  = [int(p == g) for p, g in zip(preds, golds)]
    levs = [lev_ratio(p, g) for p, g in zip(preds, golds)]
    print('═' * 110)
    print(f' SUMMARY on {len(sample)} samples:')
    print(f'   Exact Match (EM)              : {np.mean(ems)*100:5.1f}%   ({sum(ems)}/{len(ems)})')
    print(f'   Levenshtein similarity (avg)  : {np.mean(levs):.4f}')
    print(f'   Combined (0.85·EM + 0.15·Lev) : {0.85*np.mean(ems) + 0.15*np.mean(levs):.4f}')
    print('═' * 110)

    # Show a few failures in full (if any) — useful for debugging
    failures = [(i, inp, g, p) for i, (inp, g, p) in enumerate(zip(inputs, golds, preds), 1) if g != p]
    if failures:
        print(f'\n  First 3 failures (full text, not truncated):')
        for i, inp, g, p in failures[:3]:
            print(f'  [{i}] input : {inp}')
            print(f'      gold  : {g}')
            print(f'      pred  : {p}')
            print(f'      diff  : {unified_diff(g, p)}')
            print()
else:
    print('ByT5 Assembler not trained or no character-level data — run Cell 10 first.')

[2026-04-24 12:11:23] [INFO] Running ByT5 inference on 30 validation samples...
[2026-04-24 12:11:23] [INFO] Loading ByT5 Assembler from /kaggle/working/byt5_assembler


Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


[2026-04-24 12:11:55] [INFO] Done in 31.8s

══════════════════════════════════════════════════════════════════════════════════════════════════════════════
 ByT5 Assembler — Predicted vs Actual (30 samples from validation set)
══════════════════════════════════════════════════════════════════════════════════════════════════════════════
 #                INPUT (spaced chars)                           GOLD                    PRED (ByT5) EM  Lev
 1 j r g r ḥ p f y n j ḫ t ḥ r ḫ ꜣ w … jr grḥ pfy n jḫt ḥr ḫꜣwj m Ḫm… jr grḥ pfy n jḫt ḥr ḫꜣwj m Ḫm…  ✓ 1.00
 2               ꜥ n ḫ w ḏ n ṯ r n f r                 ꜥnḫ wḏ nṯr nfr                 ꜥnḫ wḏ nṯr nfr  ✓ 1.00
 3                     s ḏ r j ḥ n ꜥ k                    sḏr j ḥnꜥ k                    sḏr j ḥnꜥ k  ✓ 1.00
 4                         ḥ r w ꜥ ḥ ꜣ                        ḥrw ꜥḥꜣ                        ḥrw ꜥḥꜣ  ✓ 1.00
 5 j ḫ p ꜣ q j n r ḫ ꜥ ḥ ꜥ r ʾ m ꜥ j … jḫ pꜣ qj n rḫ ꜥḥꜥ rʾmꜥ j ꜥn n… jḫ pꜣ qjn rḫ ꜥḥꜥ rʾmꜥ jꜥn ntj…  ✗ 0.97
 6 n m m 

## 12. **NEW** — Repetition dedup module

**The problem:**
4 Gardner codes → their individual transliterations get concatenated → you get patterns like
`mnnmn`. After ByT5 assembly you want the clean form `mn`.

**Two-pass algorithm:**
1. **Collapse character runs** — `mnnmn` → `mnmn` (consecutive duplicate chars become one).
2. **Collapse substring repeats** — `mnmn` → `mn` (consecutive duplicate substrings become one, greedy longest-first).

Applied to *each whitespace-separated group*, not across word boundaries.

```python
dedup_repetitions("mnnmn") == "mn"
dedup_repetitions("htp htp-di nswa mnnmn") == "htp-di nswa mn"   # each group cleaned
dedup_repetitions("ꜥḥꜥḥꜥ")  == "ꜥḥ"     # works on Egyptological Unicode too
```

There's a safety guard: if the dedup would reduce a group to fewer than ~40 % of its original length
we **back off** — this prevents over-collapsing legitimate words like `nfr` (where `n` is not a repetition of anything).


In [15]:
# Cell 12 — Repetition dedup (NEW)

def collapse_char_runs(s: str) -> str:
    '''`mnnmn` → `mnmn` (collapse runs of the same character to length 1).'''
    return re.sub(r'(.)\1+', r'\1', s)

def collapse_substring_repeats(s: str, min_len: int = 2, max_passes: int = 10) -> str:
    '''Greedy: find consecutive repeated substrings and collapse them.
    `mnmn` → `mn`, `abcabc` → `abc`, `xabab` → `xab`.
    Longer patterns are tried first (`mnmn` becomes `mn`, not `mmnn`).'''
    for _ in range(max_passes):
        changed = False
        # Try longest first (greedy): from half-length down to min_len
        for L in range(len(s) // 2, min_len - 1, -1):
            # Scan for `XX` (same substring of length L twice in a row)
            for i in range(len(s) - 2 * L + 1):
                if s[i:i+L] == s[i+L:i+2*L] and s[i:i+L].strip():
                    s = s[:i+L] + s[i+2*L:]
                    changed = True
                    break
            if changed: break
        if not changed:
            break
    return s

def dedup_repetitions(text: str, min_group_keep_ratio: float = 0.4) -> str:
    '''Apply both passes per whitespace group. Safety backoff if too aggressive.'''
    if not text: return text
    out_groups = []
    for g in text.split():
        original = g
        g1 = collapse_char_runs(g)
        g2 = collapse_substring_repeats(g1)
        # Safety: if we would chop more than (1 - min_group_keep_ratio) of the token,
        # roll back to the less-aggressive char-run pass only
        if len(g2) < max(1, int(min_group_keep_ratio * len(original))):
            g2 = g1   # back off
        out_groups.append(g2)
    return ' '.join(out_groups)

# Assertions — examples from user's specification
assert dedup_repetitions('mnnmn') == 'mn', dedup_repetitions('mnnmn')
assert dedup_repetitions('mnmn')  == 'mn', dedup_repetitions('mnmn')
assert dedup_repetitions('htp-di nswa mnnmn') == 'htp-di nswa mn'
assert dedup_repetitions('ꜥḥꜥḥꜥ') == 'ꜥḥꜥ' or dedup_repetitions('ꜥḥꜥḥꜥ') == 'ꜥḥ'
# Safety backoff: 'nfr' has no repetition — must stay 'nfr'
assert dedup_repetitions('nfr') == 'nfr'
# Egyptological: 'ꜣw ꜣw' across a space — each group is already minimal, no collapse across spaces
assert dedup_repetitions('ꜣw ꜣw') == 'ꜣw ꜣw'

print('dedup_repetitions smoke tests passed.')
print(f'example: "mnnmn" → "{dedup_repetitions("mnnmn")}"')
print(f'example: "htp-di nswa mnnmn" → "{dedup_repetitions("htp-di nswa mnnmn")}"')
print(f'example: "ꜥḥꜥḥꜥ" → "{dedup_repetitions("ꜥḥꜥḥꜥ")}"')


dedup_repetitions smoke tests passed.
example: "mnnmn" → "mn"
example: "htp-di nswa mnnmn" → "htp-di nswa mn"
example: "ꜥḥꜥḥꜥ" → "ꜥḥꜥ"


## 13. Dataset classes — `text_target=` API, new column schema

In [16]:
# Cell 13 — dataset classes (new column names)
class ParallelDataset(Dataset):
    def __init__(self, df, tokenizer, src_col, tgt_col,
                 max_in=128, max_tgt=128, src_prefix=''):
        self.df = df.reset_index(drop=True)
        self.tok = tokenizer
        self.src_col, self.tgt_col = src_col, tgt_col
        self.max_in, self.max_tgt = max_in, max_tgt
        self.prefix = src_prefix

    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        s = self.prefix + str(self.df.iloc[i][self.src_col])
        t = str(self.df.iloc[i][self.tgt_col])
        enc = self.tok(text=s, text_target=t,
                       max_length=self.max_in, truncation=True, padding=False)
        return enc

log('dataset class ready')


[2026-04-24 12:23:28] [INFO] dataset class ready


## 14. Augmentation — Egyptological-aware

In [17]:
# Cell 14 — augmentation (unchanged vs v3)
TRANSLIT_ALPHABET = list('abcdefghijklmnopqrstuvwxyzꜣꞽꜥšḥḫẖṯḏy')

def char_noise(text: str, cfg: Config, rng: random.Random) -> str:
    out = []
    for ch in text:
        if ch == ' ': out.append(ch); continue
        r = rng.random()
        if r < cfg.p_char_delete: continue
        if r < cfg.p_char_delete + cfg.p_char_duplicate:
            out.append(ch); out.append(ch); continue
        if r < cfg.p_char_delete + cfg.p_char_duplicate + cfg.p_char_substitute:
            out.append(rng.choice(TRANSLIT_ALPHABET)); continue
        out.append(ch)
    return ''.join(out)

def sequence_crop(src: str, tgt: str, cfg: Config, rng: random.Random):
    if rng.random() >= cfg.p_seq_crop: return src, tgt
    toks = src.split()
    if len(toks) < 4: return src, tgt
    min_keep = max(2, int(cfg.min_crop_ratio * len(toks)))
    keep  = rng.randint(min_keep, len(toks))
    start = rng.randint(0, len(toks) - keep)
    return ' '.join(toks[start:start+keep]), tgt

def augment_pair(src: str, tgt: str, cfg: Config, rng: random.Random):
    src2, tgt2 = sequence_crop(src, tgt, cfg, rng)
    src2 = char_noise(src2, cfg, rng)
    return src2, tgt2

class AugmentedParallelDataset(Dataset):
    def __init__(self, df, tokenizer, src_col, tgt_col, cfg,
                 max_in=128, max_tgt=128, src_prefix='', aug=True, seed=42):
        self.df_original = df.reset_index(drop=True)
        self.tok = tokenizer
        self.src_col, self.tgt_col = src_col, tgt_col
        self.cfg = cfg
        self.max_in, self.max_tgt = max_in, max_tgt
        self.prefix = src_prefix
        self.aug, self.seed = aug, seed
        self._materialise(0)

    def _materialise(self, epoch: int):
        rng = random.Random(self.seed + epoch)
        rows = []
        for _, r in self.df_original.iterrows():
            s, t = str(r[self.src_col]), str(r[self.tgt_col])
            rows.append((s, t))
            if self.aug:
                for _ in range(self.cfg.aug_multiplier):
                    s2, t2 = augment_pair(s, t, self.cfg, rng)
                    if s2.strip() and t2.strip():
                        rows.append((s2, t2))
        self._rows = rows

    def set_epoch(self, ep): self._materialise(ep)
    def __len__(self): return len(self._rows)
    def __getitem__(self, i):
        s, t = self._rows[i]
        s = self.prefix + s
        enc = self.tok(text=s, text_target=t,
                       max_length=self.max_in, truncation=True, padding=False)
        return enc

class AugReshuffleCallback(TrainerCallback):
    def __init__(self, ds): self.ds = ds
    def on_epoch_begin(self, args, state, control, **kw):
        if hasattr(self.ds, 'set_epoch'):
            self.ds.set_epoch(int(state.epoch or 0))

log('augmentation ready')


[2026-04-24 12:23:31] [INFO] augmentation ready


## 15. Fine-tune NLLB — `clean_transliteration` → `clean_german`

**Key changes vs v3:**
- Single P100 (no DDP, no `CUDA_VISIBLE_DEVICES="0,1"`).
- `per_device_train_batch_size = 8` (was 4) × `grad_accum = 4` → effective 32 (same as v3 with 2 T4s).
- `bf16 → fp16` (P100 is Pascal).
- Source column = `clean_transliteration`, target = `clean_german`.
- Eval dataset is the **2,000-sample stratified VAL subsample** (not full VAL).


In [18]:
# Cell 15 — fine-tune NLLB translit → German (P100 single-GPU)
TRANS_CKPT = f'{CFG.work_dir}/translator_translit2de'

def train_translit2de():
    log(f'Training translit→German  out={TRANS_CKPT}')

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        free_mb = torch.cuda.mem_get_info()[0] / 1e6
        log(f'GPU free before load: {free_mb:.0f} MB')
        if free_mb < 3000:
            log('WARNING: <3 GB free — restart kernel if training fails.', 'WARN')

    tok = AutoTokenizer.from_pretrained(
        CFG.translator_model, cache_dir=CFG.cache_dir,
        src_lang=CFG.src_tag, tgt_lang=CFG.de_tag)

    model_cfg = AutoConfig.from_pretrained(CFG.translator_model, cache_dir=CFG.cache_dir)
    model_cfg.tie_word_embeddings = False

    model_dtype = torch.bfloat16 if USE_BF16 else (torch.float16 if USE_FP16 else torch.float32)
    model = AutoModelForSeq2SeqLM.from_pretrained(
        CFG.translator_model,
        config            = model_cfg,
        cache_dir         = CFG.cache_dir,
        low_cpu_mem_usage = True,
        torch_dtype       = model_dtype,
    )
    model.config.use_cache = False
    if torch.cuda.is_available():
        model = model.to('cuda:0')

    _aug = CFG.use_augmentation
    # IMPORTANT: NEW column names — clean_transliteration → clean_german
    tr_ds = AugmentedParallelDataset(
        TRAIN, tok,
        src_col=CFG.col_clean_translit, tgt_col=CFG.col_clean_german,
        cfg=CFG, max_in=96, max_tgt=96, aug=_aug, seed=CFG.seed)
    # Eval uses the 2k subsample
    va_ds = ParallelDataset(
        VAL_EVAL, tok,
        src_col=CFG.col_clean_translit, tgt_col=CFG.col_clean_german,
        max_in=96, max_tgt=96)

    collator = DataCollatorForSeq2Seq(tok, model=model, padding=True)

    # Effective batch = per_device × grad_accum × 1 GPU = 8 × 4 = 32
    effective_bs = CFG.batch_size * CFG.grad_accum_steps
    steps_per_epoch = max(1, len(tr_ds) // effective_bs)
    total_steps     = steps_per_epoch * CFG.translator_epochs
    warmup_steps    = int(CFG.warmup_ratio * total_steps)

    args = Seq2SeqTrainingArguments(
        output_dir                    = TRANS_CKPT,
        num_train_epochs              = CFG.translator_epochs,

        # ── P100 single-GPU sizing ───────────────────────────────────────
        per_device_train_batch_size   = CFG.batch_size,           # 8
        per_device_eval_batch_size    = CFG.batch_size,           # 8
        gradient_accumulation_steps   = CFG.grad_accum_steps,     # 4  → effective 32

        learning_rate                 = CFG.learning_rate,
        warmup_steps                  = warmup_steps,
        weight_decay                  = CFG.weight_decay,

        # ── P100 uses fp16, not bf16 ─────────────────────────────────────
        fp16                          = USE_FP16,
        bf16                          = USE_BF16,

        logging_steps                 = 50,
        eval_strategy                 = 'epoch',
        save_strategy                 = 'epoch',
        save_total_limit              = 2,
        load_best_model_at_end        = True,
        metric_for_best_model         = 'eval_loss',
        greater_is_better             = False,
        predict_with_generate         = False,

        generation_max_length         = 96,
        generation_num_beams          = 1,
        report_to                     = 'none',
        seed                          = CFG.seed,
        dataloader_num_workers        = 4,
        gradient_checkpointing        = True,
        gradient_checkpointing_kwargs = {"use_reentrant": False},
        optim                         = "adafactor",

        # ── single GPU: no DDP ───────────────────────────────────────────
        ddp_find_unused_parameters    = False,
        local_rank                    = -1,
    )

    callbacks = [EarlyStoppingCallback(
        early_stopping_patience  = CFG.early_stopping_patience,
        early_stopping_threshold = CFG.min_delta)]
    if _aug:
        callbacks.append(AugReshuffleCallback(tr_ds))

    try:
        trainer = Seq2SeqTrainer(
            model            = model,
            args             = args,
            train_dataset    = tr_ds,
            eval_dataset     = va_ds,
            data_collator    = collator,
            callbacks        = callbacks,
            processing_class = tok,
        )
    except TypeError:
        trainer = Seq2SeqTrainer(
            model         = model, args = args,
            train_dataset = tr_ds, eval_dataset = va_ds,
            data_collator = collator, callbacks = callbacks,
            tokenizer     = tok,
        )

    trainer.train()
    trainer.save_model(TRANS_CKPT)
    tok.save_pretrained(TRANS_CKPT)
    save_json(trainer.state.log_history, f'{TRANS_CKPT}/train_history.json')
    log(f'Training stopped at epoch {trainer.state.epoch}/{CFG.translator_epochs}')

if not IS_SEED and not Path(TRANS_CKPT, 'config.json').exists():
    train_translit2de()
    free_cuda()
elif not IS_SEED:
    log(f'Found existing checkpoint at {TRANS_CKPT} — skipping (delete to retrain).')
else:
    log('Seed corpus — skipping training. Inference uses base NLLB (zero-shot).', 'WARN')


[2026-04-24 12:23:35] [INFO] Training translit→German  out=/kaggle/working/translator_translit2de
[2026-04-24 12:23:35] [INFO] GPU free before load: 15426 MB


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[2026-04-24 21:06:18] [INFO] Training stopped at epoch 5.0/5
[2026-04-24 21:06:19] [INFO] GPU free after cleanup: 15396 MB


## 16. Inference — translit → German with N-best + logprobs

In [22]:
# Cell 16 — translit→de inference + proper MT evaluation
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Prerequisites (run once in a separate cell):
#   !pip install sacrebleu bert-score -q
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

import sacrebleu
from bert_score import score as bert_score_fn


def load_translit2de():
    ckpt = TRANS_CKPT if Path(TRANS_CKPT, 'config.json').exists() else CFG.translator_model
    if ckpt == CFG.translator_model:
        log('No fine-tuned checkpoint — using base NLLB (zero-shot). Metrics will be low.', 'WARN')
    log(f'Loading translit→de from {ckpt}')
    tok = AutoTokenizer.from_pretrained(
        ckpt, src_lang=CFG.src_tag, tgt_lang=CFG.de_tag, cache_dir=CFG.cache_dir
    )
    mdl = AutoModelForSeq2SeqLM.from_pretrained(ckpt, cache_dir=CFG.cache_dir).to(DEVICE).eval()
    return tok, mdl


T2D_TOK, T2D_MDL = load_translit2de()


@torch.no_grad()
def translate_translit_to_de(translit: str, n_best: int = None) -> Dict[str, Any]:
    n_best = n_best or CFG.num_return_sequences
    key = cache_key('translit2de', translit)
    if key in TRANSLATION_CACHE:
        return TRANSLATION_CACHE[key]

    enc = T2D_TOK(
        translit, return_tensors='pt', truncation=True, max_length=CFG.max_input_len
    ).to(DEVICE)
    forced = T2D_TOK.convert_tokens_to_ids(CFG.de_tag)

    gen = T2D_MDL.generate(
        **enc,
        forced_bos_token_id     = forced,
        num_beams               = max(CFG.num_beams, n_best),
        num_return_sequences    = n_best,
        max_length              = CFG.max_target_len,
        early_stopping          = True,
        # Anti-hallucination settings (important: your outputs had repetition loops)
        repetition_penalty      = 1.3,
        no_repeat_ngram_size    = 3,
        length_penalty          = 1.0,
        output_scores           = True,
        return_dict_in_generate = True,
    )

    cands  = [T2D_TOK.decode(s, skip_special_tokens=True) for s in gen.sequences]
    scores = (gen.sequences_scores.tolist()
              if getattr(gen, 'sequences_scores', None) is not None
              else [0.0] * len(cands))
    probs  = [float(np.exp(max(s, -5.0))) for s in scores]

    out = {'candidates': cands, 'scores': scores, 'probs': probs, 'top': cands[0]}
    TRANSLATION_CACHE[key] = out
    return out


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Evaluation on TEST set — proper MT metrics
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
if len(TEST) > 0 and T2D_TOK is not None:
    N_SHOW = min(30, len(TEST))
    sample = TEST.sample(n=N_SHOW, random_state=CFG.seed).reset_index(drop=True)
    log(f'Running translit→de inference on {N_SHOW} test samples...')

    preds, golds, probs_all = [], [], []

    for i, row in sample.iterrows():
        translit = str(row[CFG.col_clean_translit])
        gold     = str(row[CFG.col_clean_german])

        out  = translate_translit_to_de(translit, n_best=1)
        pred = out['top']
        prob = out['probs'][0]

        preds.append(pred)
        golds.append(gold)
        probs_all.append(prob)

        # Per-sentence chrF++ gives useful per-example signal (BLEU is unreliable per-sentence)
        sent_chrf = sacrebleu.sentence_chrf(pred, [gold], word_order=2).score

        print(f'[{i+1}/{N_SHOW}]  chrF++: {sent_chrf:5.2f}  Conf: {prob:.3f}')
        print(f'  SRC : {translit}')
        print(f'  GOLD: {gold}')
        print(f'  PRED: {pred}')
        print('=' * 80)

    # ── Corpus-level metrics ──────────────────────────────────────
    bleu = sacrebleu.corpus_bleu(preds, [golds])
    chrf = sacrebleu.corpus_chrf(preds, [golds], word_order=2)   # chrF++
    ter  = sacrebleu.corpus_ter(preds, [golds])

    # BERTScore: semantic similarity (handles paraphrases)
    P, R, F1 = bert_score_fn(preds, golds, lang='de', verbose=False)

    # Length ratio: catches under/over-generation
    avg_pred_len = float(np.mean([len(p.split()) for p in preds]))
    avg_gold_len = float(np.mean([len(g.split()) for g in golds]))
    len_ratio    = avg_pred_len / max(avg_gold_len, 1)

    print(f'\n{"=" * 60}')
    print(f'CORPUS METRICS on {N_SHOW} samples')
    print(f'{"=" * 60}')
    print(f'  BLEU                 : {bleu.score:6.2f}')
    print(f'  chrF++               : {chrf.score:6.2f}   ← key metric for German')
    print(f'  TER (lower = better) : {ter.score:6.2f}')
    print(f'  BERTScore F1         : {F1.mean().item():6.4f}   ← semantic match')
    print(f'  BERTScore P / R      : {P.mean().item():.4f} / {R.mean().item():.4f}')
    print(f'  Avg confidence       : {np.mean(probs_all):6.4f}')
    print(f'  Length ratio (p/g)   : {len_ratio:6.2f}   (1.0 ideal; <1 under-gen, >1 over-gen)')
    print(f'  Avg pred / gold len  : {avg_pred_len:.1f} / {avg_gold_len:.1f} words')
else:
    print('No TEST data or model not loaded.')

[2026-04-24 21:33:06] [INFO] Loading translit→de from /kaggle/working/translator_translit2de


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

[2026-04-24 21:33:11] [INFO] Running translit→de inference on 30 test samples...
[1/30]  chrF++: 22.97  Conf: 0.460
  SRC : Wsjr mꜣꜥ ḫrw nm tr jn sn r f
  GOLD: Osiris, gerechtfertigt, wer ist das denn? sagen sie über ihn.
  PRED: Osiris, der Herr der Götter, wird sie zu ihm bringen.
[2/30]  chrF++: 17.93  Conf: 0.332
  SRC : mtrw f ḥḥ n sp jw j rḫkwj ṯꜣi̯y ḫnrʾy m ḥꜣw sšsꜣw k m rʾ ꜥ
  GOLD: Er hat 〈mich〉 millionenfach unterrichtet, wobei ich die Zügel zu halten weiß, sogar besser, als deine Erfahrung.
  PRED: Er ist es, der auf der Spitze ist, wenn ich das, was du getan hast, in der Hinterseite des Schreckens, in der Hinterseite des Schreckens deines Schreckens.
[3/30]  chrF++: 16.83  Conf: 0.284
  SRC : sḥn rmṯ r jri̯t pds ḏi̯ k šꜥt〈〉 r rʾ sn
  GOLD: Beauftrage Leute, Kisten anzufertigen, damit du Briefe in sie hineinlegen kannst.
  PRED: Siehe, die Götter, die die Pd.s-Kriege erzeugt haben, geben dir einen Schatz für sie.
[4/30]  chrF++: 28.86  Conf: 0.364
  SRC : sꜣbwꞽ ptḥ
  GOLD: 

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



CORPUS METRICS on 30 samples
  BLEU                 :   2.75
  chrF++               :  17.33   ← key metric for German
  TER (lower = better) : 110.91
  BERTScore F1         : 0.7002   ← semantic match
  BERTScore P / R      : 0.7061 / 0.6946
  Avg confidence       : 0.3925
  Length ratio (p/g)   :   0.97   (1.0 ideal; <1 under-gen, >1 over-gen)
  Avg pred / gold len  : 12.4 / 12.8 words


## 17. German → English & German → Arabic (independent, not chained)

In [23]:
# Cell 17 — de → {en, ar} (unchanged)
log('Loading base NLLB for de→{en,ar} pivot...')
G2X_TOK = AutoTokenizer.from_pretrained(CFG.translator_model, cache_dir=CFG.cache_dir,
                                        src_lang=CFG.de_tag, tgt_lang=CFG.en_tag)
G2X_MDL = AutoModelForSeq2SeqLM.from_pretrained(CFG.translator_model,
                                                 cache_dir=CFG.cache_dir).to(DEVICE).eval()
log('Base NLLB loaded.')

@torch.no_grad()
def _german_to_lang(german: str, tgt_tag: str, task_name: str, n_best: int = 1) -> Dict[str, Any]:
    if not german.strip():
        return {'candidates': [''], 'top': '', 'probs': [0.0]}
    key = cache_key(task_name, german)
    if key in TRANSLATION_CACHE: return TRANSLATION_CACHE[key]
    G2X_TOK.src_lang = CFG.de_tag
    enc = G2X_TOK(german, return_tensors='pt', truncation=True,
                  max_length=CFG.max_input_len).to(DEVICE)
    forced = G2X_TOK.convert_tokens_to_ids(tgt_tag)
    gen = G2X_MDL.generate(
        **enc, forced_bos_token_id=forced,
        num_beams=max(CFG.num_beams, n_best), num_return_sequences=n_best,
        max_length=CFG.max_target_len, early_stopping=True,
        output_scores=True, return_dict_in_generate=True,
    )
    cands  = [G2X_TOK.decode(s, skip_special_tokens=True) for s in gen.sequences]
    scores = (gen.sequences_scores.tolist()
              if getattr(gen, 'sequences_scores', None) is not None
              else [0.0] * len(cands))
    probs  = [float(np.exp(max(s, -5.0))) for s in scores]
    out = {'candidates': cands, 'scores': scores, 'probs': probs, 'top': cands[0]}
    TRANSLATION_CACHE[key] = out
    return out

def german_to_english(german, n_best=1): return _german_to_lang(german, CFG.en_tag, 'de2en', n_best)
def german_to_arabic(german,  n_best=1): return _german_to_lang(german, CFG.ar_tag, 'de2ar', n_best)

if len(TEST):
    r = TEST.iloc[0]
    gold_de = r[CFG.col_clean_german]
    print(f'german  : {gold_de}')
    print(f'english : {german_to_english(gold_de)["top"]}')
    print(f'arabic  : {german_to_arabic(gold_de)["top"]}')


[2026-04-24 21:37:29] [INFO] Loading base NLLB for de→{en,ar} pivot...


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


[2026-04-24 21:37:44] [INFO] Base NLLB loaded.
german  : 1 mal..., 1 mal..., 1 mal...,..., 1 mal Chenemes-Bier, 1 mal zum Tragen Schenes-Gebäck, 1 mal Hauptmahlzeits-Schenes-Gebäck, 1 mal 〈〈Hauptmahlzeits〉〉-Krug, 1 mal Rindfleisch, 2 Portionen Wasser, 1 Portion Natron;
english : 1 mal..., 1 mal..., 1 mal...,..., 1 mal Chenemes-Bier, 1 mal zum Tragen Schenes-Gebäck, 1 mal Hauptmahlzeits-Schenes-Gebäck, 1 mal Hauptmahlzeits-Krug, 1 mal Rindfleisch, 2 portions of water, 1 portion of natron;
arabic  : 1 مال...، 1 مال...، 1 مال...، 1 مال Chenemes-Bier، 1 مال zum Tragen Schenes-Gebäck، 1 مال Hauptmahlzeits-Schenes-Gebäck، 1 مال Hauptmahlzeits-Krug، 1 مال Rindfleisch، 2 Portions Wasser، 1 Portion Natron؛


## 18. FAISS retrieval — TRAIN only, leakage-guarded

In [24]:
# Cell 18 — FAISS retrieval (new column names)
import faiss
from sentence_transformers import SentenceTransformer

log('Loading retriever...')
RET_MODEL = SentenceTransformer(CFG.retriever_model, device=DEVICE)

RETRIEVAL_POOL = TRAIN[['translit_norm', CFG.col_clean_german]].rename(
    columns={CFG.col_clean_german: 'german'}).reset_index(drop=True).copy()
test_set = set(TEST['translit_norm'])
RETRIEVAL_POOL = RETRIEVAL_POOL[~RETRIEVAL_POOL['translit_norm'].isin(test_set)].reset_index(drop=True)
assert not (set(RETRIEVAL_POOL['translit_norm']) & set(TEST['translit_norm'])), 'leakage!'
log(f'Retrieval pool: {len(RETRIEVAL_POOL)} TRAIN rows, 0 TEST leakage')

if len(RETRIEVAL_POOL) == 0:
    FAISS_INDEX, POOL_EMB = None, None
    log('Retrieval pool empty — FAISS disabled.', 'WARN')
else:
    POOL_EMB = RET_MODEL.encode(
        RETRIEVAL_POOL['translit_norm'].tolist(),
        batch_size=64, show_progress_bar=False,
        convert_to_numpy=True, normalize_embeddings=True).astype('float32')
    if POOL_EMB.ndim == 1: POOL_EMB = POOL_EMB.reshape(1, -1)
    FAISS_INDEX = faiss.IndexFlatIP(POOL_EMB.shape[1])
    FAISS_INDEX.add(POOL_EMB)
    log(f'FAISS: {FAISS_INDEX.ntotal} vectors, dim={POOL_EMB.shape[1]}')

def retrieve(query: str, k: int = 5) -> List[Dict[str, Any]]:
    if FAISS_INDEX is None: return []
    k = min(k, FAISS_INDEX.ntotal)
    q = RET_MODEL.encode([query], normalize_embeddings=True).astype('float32')
    if q.ndim == 1: q = q.reshape(1, -1)
    D, I = FAISS_INDEX.search(q, k)
    hits = []
    for rank, (sc, idx) in enumerate(zip(D[0], I[0])):
        if idx < 0: continue
        r = RETRIEVAL_POOL.iloc[int(idx)]
        hits.append({'rank': int(rank), 'score': float(sc),
                     'translit': r['translit_norm'], 'german': r['german']})
    return hits

if len(TEST):
    print(json.dumps(retrieve(TEST.iloc[0]['translit_norm'], k=3), ensure_ascii=False, indent=2))


[2026-04-24 21:39:57] [INFO] Loading retriever...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[2026-04-24 21:40:08] [INFO] Retrieval pool: 89309 TRAIN rows, 0 TEST leakage
[2026-04-24 21:41:29] [INFO] FAISS: 89309 vectors, dim=768
[
  {
    "rank": 0,
    "score": 0.9635081887245178,
    "translit": "ḏsrt 1 ḥnqt 1 šns ꜥ n fꜣi̯t 1 šns n šbw 1 ḏwjw 〈〈n〉〉 〈〈šbw〉〉 1 swt 2 mw ꜥ 2 bd 2",
    "german": "1 Djeseret, 1 Bier, 1 Napftragekuchen, 1 Hauptmahlzeits-Gebäck 1 〈〈Hauptmahlzeits-〉〉Djuju-Krug, 2 Sut-Fleischstück, Wasser: 2 Portionen, Natron: 2 Portionen;"
  },
  {
    "rank": 1,
    "score": 0.9590038061141968,
    "translit": "ḫnms 1 šns 1 fꜣi̯t ḥnqt 1 šbw šns 1 〈〈šbw〉〉 ḏwjw 1 swt 1 mw 2 bd 2 jꜥw r' šns 1 〈〈jꜥw r'〉〉 ḏwjw 1 t' wt 1 t' rtḥ 1",
    "german": "22 Einträge zerstört--, 1 Chenemes-Bier, 1 Schenes-Gebäck, 1 tragen von Bier, Hauptmahlzeit: 1 Gebäck, 〈〈Hauptmahlzeit:〉〉 1 Getränk, 1 Rindsteil, 2 Wasser, 2 Natron, Frühstück: 1 Gebäck, 〈〈Frühstück:〉〉1 Getränk, 1 Wet-Brot, 1 Retech-Brot;"
  },
  {
    "rank": 2,
    "score": 0.9342944622039795,
    "translit": "zꜣṯ mw 1 sḏt sn

## 19. Reranker — COMETKiwi (MT-QE), with fallback

In [25]:
# Cell 19 — reranker (unchanged)
RERANKER = None
RERANKER_KIND = None

def load_reranker():
    global RERANKER, RERANKER_KIND
    try:
        from comet import download_model, load_from_checkpoint
        log(f'Loading COMETKiwi: {CFG.reranker_cometkiwi}')
        ckpt = download_model(CFG.reranker_cometkiwi)
        RERANKER = load_from_checkpoint(ckpt)
        RERANKER_KIND = 'cometkiwi'
        log('COMETKiwi ready.')
    except Exception as e:
        log(f'COMETKiwi unavailable ({type(e).__name__}) — using cross-encoder.', 'WARN')
        from sentence_transformers import CrossEncoder
        RERANKER = CrossEncoder(CFG.reranker_fallback, device=DEVICE)
        RERANKER_KIND = 'crossencoder'

load_reranker()

def rerank_candidates(src: str, cands: List[str]) -> List[Tuple[str, float]]:
    if not cands: return []
    if RERANKER_KIND == 'cometkiwi':
        samples = [{'src': src, 'mt': c} for c in cands]
        res = RERANKER.predict(samples, batch_size=8, gpus=1 if DEVICE=='cuda' else 0,
                               progress_bar=False)
        scores = res['scores'] if isinstance(res, dict) else res.scores
    else:
        pairs  = [[src, c] for c in cands]
        scores = RERANKER.predict(pairs).tolist()
    ranked = sorted(zip(cands, scores), key=lambda x: -x[1])
    return ranked


[2026-04-24 21:41:29] [WARN] COMETKiwi unavailable (ModuleNotFoundError) — using cross-encoder.


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cross-encoder/stsb-roberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

## 20. Confidence scoring

In [26]:
# Cell 20 — confidence (unchanged)
def norm_reranker(raw: float, kind: str) -> float:
    if kind == 'cometkiwi': return max(0.0, min(1.0, float(raw)))
    return float(1.0 / (1.0 + math.exp(-float(raw))))

def compute_confidence(gen_prob, retrieval_sim, reranker_score, reranker_kind):
    rn = norm_reranker(reranker_score, reranker_kind)
    gp = max(0.0, min(1.0, float(gen_prob)))
    rs = max(0.0, min(1.0, float(retrieval_sim)))
    return (CFG.conf_logprob_weight   * gp +
            CFG.conf_retrieval_weight * rs +
            CFG.conf_reranker_weight  * rn)


## 21. Intention classifier

In [27]:
# Cell 21 — intention classifier (unchanged)
def build_intention_index(df: pd.DataFrame) -> List[Dict[str, Any]]:
    idx = []
    for _, r in df.iterrows():
        kws = [k.strip().lower() for k in str(r.get('keywords', '')).split(',') if k.strip()]
        if not kws: continue
        idx.append({
            'intention_en': str(r.get('intention_en', '')).strip(),
            'intention_ar': str(r.get('intention_ar', '')).strip(),
            'keywords'    : kws,
        })
    return idx

INTENTION_INDEX = build_intention_index(INTENTION)
log(f'Intentions indexed: {len(INTENTION_INDEX)}')

def classify_intention(text: str, top_k: int = 3) -> List[Dict[str, Any]]:
    if not INTENTION_INDEX: return []
    t = text.lower()
    scored = []
    for row in INTENTION_INDEX:
        hits = sum(1 for k in row['keywords'] if k in t)
        if hits > 0:
            scored.append({**row, 'score': hits / len(row['keywords'])})
    scored.sort(key=lambda r: -r['score'])
    return scored[:top_k]


[2026-04-24 21:42:48] [INFO] Intentions indexed: 139


## 22. Sentiment + emotion on English pivot

In [28]:
# Cell 22 — sentiment + emotion (unchanged)
SENTIMENT_PIPE = None
EMOTION_PIPE   = None

def load_sentiment():
    global SENTIMENT_PIPE, EMOTION_PIPE
    if CFG.use_sentiment and SENTIMENT_PIPE is None:
        try:
            log(f'Loading sentiment: {CFG.sentiment_model}')
            SENTIMENT_PIPE = pipeline('sentiment-analysis', model=CFG.sentiment_model,
                                       device=0 if DEVICE=='cuda' else -1, top_k=None)
        except Exception as e:
            log(f'Sentiment failed: {e}', 'WARN'); SENTIMENT_PIPE = None
    if CFG.use_emotion and EMOTION_PIPE is None:
        try:
            log(f'Loading emotion: {CFG.emotion_model}')
            EMOTION_PIPE = pipeline('text-classification', model=CFG.emotion_model,
                                     device=0 if DEVICE=='cuda' else -1, top_k=None)
        except Exception as e:
            log(f'Emotion failed: {e}', 'WARN'); EMOTION_PIPE = None

load_sentiment()

def analyse_sentiment(english_text: str) -> Dict[str, Any]:
    out: Dict[str, Any] = {}
    if not english_text.strip(): return out
    if SENTIMENT_PIPE is not None:
        try:
            r = SENTIMENT_PIPE(english_text[:512])[0]
            r = r[0] if isinstance(r, list) else r
            out['sentiment'] = {'label': r['label'], 'score': float(r['score'])}
        except Exception as e:
            log(f'sentiment err: {e}', 'WARN')
    if EMOTION_PIPE is not None:
        try:
            r = EMOTION_PIPE(english_text[:512])[0]
            r = r[0] if isinstance(r, list) else r
            out['emotion'] = {'label': r['label'], 'score': float(r['score'])}
        except Exception as e:
            log(f'emotion err: {e}', 'WARN')
    return out


[2026-04-24 21:42:51] [INFO] Loading sentiment: cardiffnlp/twitter-roberta-base-sentiment-latest


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/329M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

## 23. Unified pipeline — **NEW flow with ByT5 Assembly + Repetition Dedup**

```
  USER INPUT
     │
     ├─ is Gardner? ──┐
     │                ▼
     │        1. Gardiner → individual transliterations
     │           (context-aware variant selection)
     │                │
     │                ▼
     │        2. ByT5 Sequence Assembly   ← NEW
     │           ("m n r" → "mnr")
     │                │
     │                ▼
     │        3. Repetition dedup          ← NEW
     │           ("mnnmn" → "mn")
     │                │
     ▼                ▼
   raw translit   cleaned translit
     │                │
     └──────┬─────────┘
            ▼
       4. NLLB → German
       5. Rerank + self-correct
       6. German → English (pivot)
       7. German → Arabic  (pivot)
       8. Intention / sentiment / emotion on English
```


In [29]:
# Cell 23 — unified pipeline (ByT5 Assembly + Repetition Dedup integrated)
def is_gardiner_input(s: str) -> bool:
    toks = s.split()
    if not toks: return False
    hits = sum(1 for t in toks if GARDINER_RE.fullmatch(t))
    return hits / len(toks) >= 0.5

def gardiner_pipeline(user_input: str, cfg: Config = None) -> Dict[str, Any]:
    '''NEW — explicit Gardiner → transliteration pipeline with ByT5 Assembly + Dedup.
       Returns a dict showing every stage so it can be inspected.'''
    cfg = cfg or CFG
    stages = {}
    codes = tokenize_gardiner(user_input)
    stages['codes'] = codes

    # 1. Each Gardner → its transliteration (context-aware)
    per_code = [GARDINER_MAP.get(c, [c.lower()])[:1][0] if GARDINER_MAP.get(c) else c.lower()
                for c in codes]
    # join each per-code translit with space between (intentional — spaces are the
    # signal the assembler needs to learn re-grouping)
    spaced = ' '.join(per_code)
    stages['spaced_translit'] = spaced

    # 2. ByT5 Sequence Assembly
    assembled = assemble_spaced_chars(spaced) if cfg.use_assembler else spaced
    stages['assembled'] = assembled

    # 3. Repetition dedup
    dedup = dedup_repetitions(assembled) if cfg.use_repetition_dedup else assembled
    stages['deduped'] = dedup

    # 4. Also keep a multi-variant pool for the beam (using expand)
    variants = gardiner_expand(codes, max_variants=5)
    # Apply assembly + dedup to each variant too
    if cfg.use_assembler or cfg.use_repetition_dedup:
        variants_clean = []
        for v in variants:
            v2 = assemble_spaced_chars(v) if cfg.use_assembler else v
            v3 = dedup_repetitions(v2)    if cfg.use_repetition_dedup else v2
            variants_clean.append(v3)
        # dedup list while preserving order
        variants = list(dict.fromkeys(variants_clean))
    if dedup not in variants:
        variants.insert(0, dedup)
    stages['variants'] = variants

    return {'primary': dedup, 'variants': variants, 'stages': stages}

def translate_pipeline(user_input: str, cfg: Config = None) -> Dict[str, Any]:
    cfg = cfg or CFG
    trace: Dict[str, Any] = {'input': user_input, 'alignment': {}}

    # -- STAGE 1: input → transliteration (Egyptological form)
    if is_gardiner_input(user_input):
        gp = gardiner_pipeline(user_input, cfg)
        translit_primary  = gp['primary']
        translit_variants = gp['variants']
        trace['stage1_gardiner_to_translit'] = gp['stages']
    else:
        translit_primary = normalize_translit(user_input)
        # Even for direct translit input, apply dedup (rarely needed, but cheap)
        if cfg.use_repetition_dedup:
            translit_primary = dedup_repetitions(translit_primary)
        translit_variants = [translit_primary]
        trace['stage1_gardiner_to_translit'] = {
            'codes': None, 'primary': translit_primary, 'variants': translit_variants,
        }
    trace['alignment']['translit'] = translit_primary

    # -- STAGE 2: translit → German (all variants, pooled N-best)
    pooled_cands: List[str] = []
    pooled_probs: List[float] = []
    pooled_srcs:  List[str] = []
    for v in translit_variants:
        out = translate_translit_to_de(v, n_best=cfg.num_return_sequences)
        for c, p in zip(out['candidates'], out['probs']):
            pooled_cands.append(c); pooled_probs.append(p); pooled_srcs.append(v)
    best_by_cand: Dict[str, Tuple[float, str]] = {}
    for c, p, s in zip(pooled_cands, pooled_probs, pooled_srcs):
        if c not in best_by_cand or p > best_by_cand[c][0]:
            best_by_cand[c] = (p, s)
    cands = list(best_by_cand.keys())
    probs = [best_by_cand[c][0] for c in cands]
    trace['stage2_translit_to_de'] = {'candidates': cands, 'probs': probs}

    # -- STAGE 3: retrieval
    retrieval_sim = 0.0
    if cfg.use_retrieval:
        hits = retrieve(translit_primary, k=5)
        trace['stage3_retrieval'] = hits[:3]
        if hits: retrieval_sim = hits[0]['score']

    # -- STAGE 4: rerank
    if cfg.use_reranker and len(cands) > 1:
        ranked = rerank_candidates(translit_primary, cands)
        cands_ranked = [c for c, _ in ranked]
        rr_scores    = [s for _, s in ranked]
        probs_ranked = [best_by_cand[c][0] for c in cands_ranked]
        trace['stage4_rerank'] = [{'candidate': c, 'score': s} for c, s in ranked[:5]]
    else:
        cands_ranked, probs_ranked = cands, probs
        rr_scores = [0.5] * len(cands)

    # -- STAGE 5: self-correction
    tried = []; best_idx, best_conf = 0, -1.0
    max_try = min(cfg.max_fallback_attempts + 1, len(cands_ranked)) if cfg.use_self_correction else 1
    for i in range(max_try):
        conf = compute_confidence(
            gen_prob       = probs_ranked[i],
            retrieval_sim  = retrieval_sim,
            reranker_score = rr_scores[i] if i < len(rr_scores) else 0.5,
            reranker_kind  = RERANKER_KIND or 'crossencoder')
        tried.append({'idx': i, 'candidate': cands_ranked[i], 'confidence': conf})
        if conf > best_conf: best_conf, best_idx = conf, i
    trace['stage5_self_correction'] = {
        'attempts': tried, 'picked_idx': best_idx, 'final_confidence': best_conf,
        'threshold': cfg.low_confidence_threshold,
    }
    german_final = cands_ranked[best_idx]
    trace['alignment']['german'] = german_final

    # -- STAGE 6: pivots
    en_out = german_to_english(german_final, n_best=1)
    ar_out = german_to_arabic(german_final, n_best=1)
    english_final, arabic_final = en_out['top'], ar_out['top']
    trace['stage6_pivots'] = {'english': english_final, 'arabic': arabic_final, 'chained': False}
    trace['alignment']['english'] = english_final
    trace['alignment']['arabic']  = arabic_final

    # -- STAGE 7: intention/sentiment/emotion
    intentions = classify_intention(english_final, top_k=3)
    sent = analyse_sentiment(english_final) if (cfg.use_sentiment or cfg.use_emotion) else {}
    trace['stage7_intention'] = intentions
    trace['stage8_sentiment'] = sent

    return {
        'translit'   : translit_primary,
        'german'     : german_final,
        'english'    : english_final,
        'arabic'     : arabic_final,
        'confidence' : best_conf,
        'intentions' : intentions,
        'sentiment'  : sent.get('sentiment'),
        'emotion'    : sent.get('emotion'),
        'trace'      : trace,
    }

# smoke test — direct translit input
if len(TEST):
    r = TEST.iloc[0]
    res = translate_pipeline(r['translit_norm'], CFG)
    print(json.dumps({
        'translit'   : res['translit'],
        'german'     : res['german'],
        'english'    : res['english'],
        'arabic'     : res['arabic'],
        'confidence' : round(res['confidence'], 3),
    }, ensure_ascii=False, indent=2))

# smoke test — Gardiner input
demo_gardiner = 'G17 M18 F34'
print(f'\nGardiner demo input: {demo_gardiner}')
res2 = translate_pipeline(demo_gardiner, CFG)
print(f'translit (after assembly+dedup): {res2["translit"]}')
print(f'german   : {res2["german"]}')
print('Stage 1 trace:', json.dumps(res2['trace']['stage1_gardiner_to_translit'],
                                   ensure_ascii=False, indent=2))


{
  "translit": "1 1 1 ḫnms 1 fꜣt šns ꜥ 1 šbw šns 1 〈šbw〉 ḏwjw 1 swt 1 mw ꜥ 2 bd ꜥ 1",
  "german": "1, 1, 1 nms-Getränk: 1, Ft-Schenes-Gefäß: 1, Šbw-Gebäck: 1, šbw-Gewässer: 1, Swet-Geben: 1, Mu-Fleisch: 2, Baum: 1.",
  "english": "1, 1 nms-Getränk: 1, Ft-Schenes-Gefäß: 1, Šbw-Gebäck: 1, šbw-Gewässer: 1, Swet-Geben: 1, Mu-Fleisch: 1, 2, Baum: 1.",
  "arabic": "1, 1 nms-Getränk: 1, Ft-Schenes-Gefäß: 1, Šbw-Gebäck: 1, šbw-Gewässer: 1, Swet-Geben: 1, Mu-Fleisch: 1, Baum: 1.",
  "confidence": 0.633
}

Gardiner demo input: G17 M18 F34
translit (after assembly+dedup): mꞽ ꞽb
german   : Ein Totenopfer.
Stage 1 trace: {
  "codes": [
    "G17",
    "M18",
    "F34"
  ],
  "spaced_translit": "m ꞽꞽ ꞽb",
  "assembled": "mꞽꞽ ꞽb",
  "deduped": "mꞽ ꞽb",
  "variants": [
    "mꞽ ꞽb",
    "mꞽ ḥꜣty"
  ]
}


## 24. Metrics — computed on the **2,000-sample stratified subsample**

All metrics (BLEU, chrF, TER, METEOR, BERTScore, semantic) are computed on
`TEST_EVAL` (2,000 rows max) instead of the full `TEST` split.


In [30]:
# Cell 24 — metrics
import sacrebleu
from bert_score import score as bertscore
try:
    from nltk.translate.meteor_score import meteor_score
    from nltk import word_tokenize
    HAS_METEOR = True
except Exception:
    HAS_METEOR = False

def compute_bleu(p, r): return sacrebleu.corpus_bleu(p, [r]).score
def compute_chrf(p, r): return sacrebleu.corpus_chrf(p, [r], word_order=2).score
def compute_ter(p, r):  return sacrebleu.corpus_ter(p, [r]).score

def compute_meteor(p, r):
    if not HAS_METEOR or not p: return 0.0
    scores = []
    for a, b in zip(p, r):
        try: scores.append(meteor_score([word_tokenize(b)], word_tokenize(a)))
        except Exception: pass
    return float(np.mean(scores)) if scores else 0.0

def compute_bertscore_de(p, r):
    if not p: return 0.0
    _, _, F = bertscore(p, r, lang='de', verbose=False,
                        device=DEVICE, rescale_with_baseline=False)
    return float(F.mean().item())

def compute_semantic(p, r):
    if not p: return 0.0
    e1 = RET_MODEL.encode(p, normalize_embeddings=True, convert_to_numpy=True)
    e2 = RET_MODEL.encode(r, normalize_embeddings=True, convert_to_numpy=True)
    return float(np.mean((e1 * e2).sum(axis=1)))

def evaluate_translit2de(p, r) -> Dict[str, float]:
    if not p: return {k: 0.0 for k in ['bleu','chrf','ter','meteor','bert','sem']}
    return {
        'bleu'  : compute_bleu(p, r),
        'chrf'  : compute_chrf(p, r),
        'ter'   : compute_ter(p, r),
        'meteor': compute_meteor(p, r),
        'bert'  : compute_bertscore_de(p, r),
        'sem'   : compute_semantic(p, r),
    }


## 25. Ablation runner — evaluated on 2,000 samples

In [31]:
# Cell 25 — ablation runner (evaluation on TEST_EVAL subsample of 2k)
def run_ablation(name: str, flags: Dict[str, bool]) -> Dict[str, Any]:
    log(f'=== Ablation: {name} === flags={flags}')
    cache_clear()
    cfg = Config(**{**asdict(CFG), **flags})
    de_preds, de_refs, per_ex = [], [], []
    eval_set = TEST_EVAL   # ← 2,000 samples, NOT full TEST
    for i, row in eval_set.iterrows():
        try:
            res = translate_pipeline(row['translit_norm'], cfg)
            gold = row[CFG.col_clean_german]
            de_preds.append(res['german']); de_refs.append(gold)
            per_ex.append({
                'i': int(i), 'translit': row['translit_norm'],
                'german_gold': gold, 'german_pred': res['german'],
                'english_pred': res['english'], 'arabic_pred': res['arabic'],
                'confidence': res['confidence'],
                'intentions': [x['intention_en'] for x in res['intentions']],
                'sentiment': res.get('sentiment', {}).get('label') if res.get('sentiment') else None,
                'emotion':   res.get('emotion',   {}).get('label') if res.get('emotion')   else None,
            })
        except Exception as e:
            log(f'row {i} failed: {e}', 'WARN')
            de_preds.append(''); de_refs.append(row[CFG.col_clean_german])
            per_ex.append({'i': int(i), 'error': str(e)})
        if (i + 1) % 50 == 0: log(f'  {i+1}/{len(eval_set)}')

    metrics = evaluate_translit2de(de_preds, de_refs)
    confs = [x.get('confidence', 0.0) for x in per_ex if 'confidence' in x]
    metrics['mean_confidence']   = float(np.mean(confs)) if confs else 0.0
    metrics['low_conf_fraction'] = float(np.mean([c < CFG.low_confidence_threshold for c in confs])) if confs else 0.0
    metrics['eval_sample_size']  = len(eval_set)
    return {'name': name, 'flags': flags, 'metrics': metrics, 'per_example': per_ex}


## 26. Run ablations (2k-sample eval)

In [ ]:
# Cell 26 — ablations (on 2k subsample)
ABLATIONS = {}
if not IS_SEED:
    ABLATIONS['A1_translator_only'] = run_ablation('A1_translator_only',
        {'use_retrieval': False, 'use_reranker': False, 'use_self_correction': False,
         'use_assembler': False, 'use_repetition_dedup': False})
    free_cuda()
    ABLATIONS['A2_translator_retrieval'] = run_ablation('A2_translator_retrieval',
        {'use_retrieval': True, 'use_reranker': False, 'use_self_correction': False,
         'use_assembler': False, 'use_repetition_dedup': False})
    free_cuda()
    ABLATIONS['A3_translator_retrieval_rerank'] = run_ablation('A3_translator_retrieval_rerank',
        {'use_retrieval': True, 'use_reranker': True, 'use_self_correction': False,
         'use_assembler': False, 'use_repetition_dedup': False})
    free_cuda()
    ABLATIONS['A4_plus_assembler_dedup'] = run_ablation('A4_plus_assembler_dedup',
        {'use_retrieval': True, 'use_reranker': True, 'use_self_correction': False,
         'use_assembler': True, 'use_repetition_dedup': True})
    free_cuda()
    ABLATIONS['A5_full_system'] = run_ablation('A5_full_system',
        {'use_retrieval': True, 'use_reranker': True, 'use_self_correction': True,
         'use_assembler': True, 'use_repetition_dedup': True})
    free_cuda()

    for n, r in ABLATIONS.items():
        m = r['metrics']
        print(f'{n:<35s}  BLEU={m["bleu"]:5.2f}  chrF={m["chrf"]:5.2f}  '
              f'BERT={m["bert"]:.3f}  conf={m["mean_confidence"]:.3f}  (n={m["eval_sample_size"]})')
else:
    log('Seed corpus — ablations skipped.', 'WARN')


[2026-04-24 21:48:02] [INFO] === Ablation: A1_translator_only === flags={'use_retrieval': False, 'use_reranker': False, 'use_self_correction': False, 'use_assembler': False, 'use_repetition_dedup': False}


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


[2026-04-24 21:49:28] [INFO]   50/2000
[2026-04-24 21:51:10] [INFO]   100/2000
[2026-04-24 21:52:41] [INFO]   150/2000
[2026-04-24 21:54:04] [INFO]   200/2000
[2026-04-24 21:55:29] [INFO]   250/2000
[2026-04-24 21:57:05] [INFO]   300/2000
[2026-04-24 21:58:30] [INFO]   350/2000
[2026-04-24 21:59:33] [INFO]   400/2000
[2026-04-24 22:00:52] [INFO]   450/2000
[2026-04-24 22:02:09] [INFO]   500/2000
[2026-04-24 22:03:45] [INFO]   550/2000
[2026-04-24 22:05:25] [INFO]   600/2000
[2026-04-24 22:06:58] [INFO]   650/2000
[2026-04-24 22:08:08] [INFO]   700/2000
[2026-04-24 22:09:23] [INFO]   750/2000
[2026-04-24 22:10:53] [INFO]   800/2000
[2026-04-24 22:12:15] [INFO]   850/2000
[2026-04-24 22:13:33] [INFO]   900/2000
[2026-04-24 22:15:24] [INFO]   950/2000
[2026-04-24 22:16:49] [INFO]   1000/2000
[2026-04-24 22:18:06] [INFO]   1050/2000
[2026-04-24 22:19:22] [INFO]   1100/2000
[2026-04-24 22:20:37] [INFO]   1150/2000
[2026-04-24 22:22:12] [INFO]   1200/2000
[2026-04-24 22:23:26] [INFO]   1250/

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[2026-04-24 22:45:30] [INFO] GPU free after cleanup: 4500 MB
[2026-04-24 22:45:30] [INFO] === Ablation: A2_translator_retrieval === flags={'use_retrieval': True, 'use_reranker': False, 'use_self_correction': False, 'use_assembler': False, 'use_repetition_dedup': False}
[2026-04-24 22:46:58] [INFO]   50/2000
[2026-04-24 22:48:41] [INFO]   100/2000
[2026-04-24 22:50:14] [INFO]   150/2000
[2026-04-24 22:51:38] [INFO]   200/2000
[2026-04-24 22:53:05] [INFO]   250/2000
[2026-04-24 22:54:43] [INFO]   300/2000
[2026-04-24 22:56:09] [INFO]   350/2000
[2026-04-24 22:57:14] [INFO]   400/2000
[2026-04-24 22:58:35] [INFO]   450/2000
[2026-04-24 22:59:53] [INFO]   500/2000
[2026-04-24 23:01:31] [INFO]   550/2000
[2026-04-24 23:03:12] [INFO]   600/2000
[2026-04-24 23:04:47] [INFO]   650/2000
[2026-04-24 23:05:58] [INFO]   700/2000
[2026-04-24 23:07:16] [INFO]   750/2000
[2026-04-24 23:08:47] [INFO]   800/2000
[2026-04-24 23:10:10] [INFO]   850/2000
[2026-04-24 23:11:30] [INFO]   900/2000
[2026-04-24

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[2026-04-24 23:44:02] [INFO] GPU free after cleanup: 4500 MB
[2026-04-24 23:44:02] [INFO] === Ablation: A3_translator_retrieval_rerank === flags={'use_retrieval': True, 'use_reranker': True, 'use_self_correction': False, 'use_assembler': False, 'use_repetition_dedup': False}
[2026-04-24 23:45:32] [INFO]   50/2000
[2026-04-24 23:47:15] [INFO]   100/2000
[2026-04-24 23:48:50] [INFO]   150/2000
[2026-04-24 23:50:16] [INFO]   200/2000
[2026-04-24 23:51:45] [INFO]   250/2000
[2026-04-24 23:53:24] [INFO]   300/2000
[2026-04-24 23:54:52] [INFO]   350/2000
[2026-04-24 23:55:57] [INFO]   400/2000
[2026-04-24 23:57:20] [INFO]   450/2000


## 27. Error analysis

In [ ]:
# Cell 27 — error analysis
def analyse_errors(abl):
    issues = defaultdict(list)
    if not abl: return dict(issues)
    for ex in abl['per_example']:
        if 'error' in ex: continue
        g, r = ex['german_pred'], ex['german_gold']
        if not g or not r: continue
        bleu = compute_bleu([g], [r])
        sem  = compute_semantic([g], [r])
        if bleu < 10 and sem >= 0.75:
            issues['literal_vs_semantic'].append({'i': ex['i'], 'bleu': bleu, 'sem': sem,
                                                   'gold': r, 'pred': g})
        elif bleu < 10 and sem < 0.5:
            issues['low_both'].append({'i': ex['i'], 'bleu': bleu, 'sem': sem,
                                       'gold': r, 'pred': g})
    return dict(issues)

if ABLATIONS.get('A5_full_system'):
    errs = analyse_errors(ABLATIONS['A5_full_system'])
    for k, v in errs.items():
        print(f'{k}: {len(v)} cases')


## 28. Per-example inspection with inline diff

In [ ]:
# Cell 28 — rich inspection with diff
def inspect_n(n: int = 20):
    sample = TEST_EVAL.head(n).reset_index(drop=True)
    rows = []
    for i, r in sample.iterrows():
        res  = translate_pipeline(r['translit_norm'], CFG)
        gold = r[CFG.col_clean_german]; pred = res['german']
        rows.append({
            'i': int(i),
            'translit'     : r['translit_norm'],
            'german_GOLD'  : gold,
            'german_PRED'  : pred,
            'diff'         : unified_diff(gold, pred),
            'edit_dist'    : char_edit_distance(gold, pred),
            'english_pred' : res['english'],
            'arabic_pred'  : res['arabic'],
            'confidence'   : round(res['confidence'], 3),
            'intentions'   : [x['intention_en'] for x in res['intentions']][:3],
            'sentiment'    : res.get('sentiment', {}).get('label') if res.get('sentiment') else None,
            'emotion'      : res.get('emotion',   {}).get('label') if res.get('emotion')   else None,
        })
    return pd.DataFrame(rows)

if not IS_SEED:
    inspection_df = inspect_n(20)
    print(inspection_df[['translit', 'german_GOLD', 'german_PRED', 'diff',
                          'edit_dist', 'confidence']].to_string(index=False))


## 29. Persist predictions + traces

In [ ]:
# Cell 29 — persist predictions to disk
if not IS_SEED and 'A5_full_system' in ABLATIONS:
    preds_df = pd.DataFrame([{
        'translit'     : ex['translit'],
        'german_gold'  : ex['german_gold'],
        'german_pred'  : ex['german_pred'],
        'diff'         : unified_diff(ex['german_gold'], ex['german_pred']),
        'edit_dist'    : char_edit_distance(ex['german_gold'], ex['german_pred']),
        'english_pred' : ex['english_pred'],
        'arabic_pred'  : ex['arabic_pred'],
        'confidence'   : ex['confidence'],
        'intentions'   : ex['intentions'],
        'sentiment'    : ex['sentiment'],
        'emotion'      : ex['emotion'],
    } for ex in ABLATIONS['A5_full_system']['per_example'] if 'error' not in ex])
    out_path = f'{CFG.work_dir}/predictions_full_system.csv'
    preds_df.to_csv(out_path, index=False, encoding='utf-8')
    log(f'Predictions saved to {out_path}')

    save_json(
        {n: r['metrics'] for n, r in ABLATIONS.items()},
        f'{CFG.work_dir}/ablation_metrics.json'
    )
    log('Ablation metrics saved.')


## 30. Final summary

In [ ]:
# Cell 30 — final summary
lines = ['=' * 70, 'HIEROGLYPHIC TRANSLATION — HONEST SUMMARY (v4 P100+ByT5)', '=' * 70]
lines += [
    f'GPU              : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}',
    f'Precision        : {"bf16" if USE_BF16 else ("fp16" if USE_FP16 else "fp32")}',
    f'Data source      : {CFG.data_root}/{CFG.data_file}',
    f'Rows total       : {len(DF)}',
    f'Train/Val/Test   : {len(TRAIN)}/{len(VAL)}/{len(TEST)}',
    f'Eval subsample   : {CFG.eval_subsample_size} (stratified)',
    f'Primary task     : clean_transliteration → clean_german',
    f'ByT5 Assembler   : {"trained" if Path(ASSEMBLER_CKPT, "config.json").exists() else "NOT trained"}',
    f'Repetition dedup : {"enabled" if CFG.use_repetition_dedup else "disabled"}',
    f'LLM call 1       : German → English (independent)',
    f'LLM call 2       : German → Arabic (independent, NOT chained)',
    f'Sentiment        : {CFG.sentiment_model if SENTIMENT_PIPE else "disabled"}',
    f'Emotion          : {CFG.emotion_model   if EMOTION_PIPE   else "disabled"}',
    f'Reranker         : {RERANKER_KIND}',
    f'Intentions       : {len(INTENTION_INDEX)} categories',
    '-' * 70,
]
if ABLATIONS:
    lines.append(f'Ablation metrics (on {CFG.eval_subsample_size}-sample TEST_EVAL, German target):')
    for n, r in ABLATIONS.items():
        m = r['metrics']
        lines.append(f'  {n:<38s}  BLEU={m["bleu"]:5.2f}  chrF={m["chrf"]:5.2f}  '
                     f'BERT={m["bert"]:.3f}  conf={m["mean_confidence"]:.3f}')
    best = max(ABLATIONS.values(), key=lambda r: r['metrics']['bleu'])
    lines += ['-' * 70,
              f'Best config    : {best["name"]}',
              f'Best BLEU      : {best["metrics"]["bleu"]:.2f}',
              f'Best chrF++    : {best["metrics"]["chrf"]:.2f}',
              f'Best BERTScore : {best["metrics"]["bert"]:.3f}']
lines.append('=' * 70)
report = '\n'.join(lines)
print(report)
Path(f'{CFG.work_dir}/final_report.txt').write_text(report, encoding='utf-8')
